In [ ]:
# smallm-125m: tokenizer, pretraining, instruction tuning. Kaggle, GPU T4 x2,# internet on, persistence "Files only". Press Run All; the plan takes about three# sessions, and each Run All continues where the last one stopped. Stop the session# by hand when a cell says so - idle time is charged to the weekly quota.## Worth changing:#   plan_h          total pretraining budget in *training hours* (26.5 = this model)#   de_ratio        share of German tokens: 0.40 here, 0.20 in POST for fine-tuning#   en_sources,     the corpus mixture; weights apply within a language, on characters#   de_sources,#   source_weights#   micro_bs,       tokens per step = micro_bs x grad_accum x 2 GPUs x seq_len#   grad_accum#   sft_tokens,     size of the chat corpus and how many passes over it#   sft_epochs#   time_budget_h   POST: wall-clock cap for fine-tuning. CONFIG: optional cap on one#                   pretraining session, None means "whatever the session has left"CONFIG = {   'n_layer': 12,    'n_head': 12,    'dim': 768,    'mlp_mult': 4,    'seq_len': 1024,    'softcap': 15.0,    'relu2_clamp': 200.0,    'vocab_size': 32768,    'tok_train_chars': 300000000,    'eot_token': '<|endoftext|>',    'en_sources': ['fineweb-edu', 'cosmopedia'],    'de_sources': ['fineweb2-hq'],    'source_weights': {'fineweb-edu': 0.5, 'cosmopedia': 0.5, 'fineweb2-hq': 1.0},    'val_files_per_source': 1,    'de_ratio': 0.4,    'min_edu_score': 0.0,    'min_chars': 200,    'min_ocr_score': 90,    'keep_null_ocr_score': False,    'chars_per_token': 4.55,    'de_streams': 3,    'de_max_chars_per_file': 80000000,    'doc_batch': 64,    'data_workers': 1,    'prefetch_factor': 32,    'anneal_frac': 0.15,    'anneal_min_edu_score': 3.5,    'anneal_sources': ['fineweb-edu', 'cosmopedia', 'fineweb2-hq'],    'anneal_source_weights': {'fineweb-edu': 0.25, 'cosmopedia': 0.75, 'fineweb2-hq': 1.0},    'micro_bs': 8,    'grad_accum': 8,    'total_steps': None,    'plan_h': 26.5,    'time_budget_h': None,    'reserve_seconds': 300,    'calibration_steps': 25,    'recalibrate_every': 300,    'warmup_steps': 150,    'cooldown_frac': 0.2,    'final_lr_frac': 0.02,    'short_seq_frac': 0.3,    'lr_muon': 0.025,    'muon_momentum': 0.95,    'ns_steps': 5,    'momentum_warmup_steps': 300,    'muon_weight_decay': 0.02,    'lr_embed': 0.3,    'lr_head': 0.004,    'lr_scalar': 0.02,    'adam_betas': (0.8, 0.95),    'grad_clip': 2.0,    'min_loss_scale': 64.0,    'max_loss_scale': 65536.0,    'recovery_loss_scale': 1024.0,    'max_consecutive_skips': 25,    'max_recoveries': 3,    'compile': True,    'compile_mode': 'default',    'coordinate_descent_tuning': False,    'fp16_allreduce': True,    'seed': 1337,    'nccl_timeout_min': 60,    'log_every': 10,    'val_every': 250,    'val_batches': 20,    'val_tokens': 2000000,    'save_every': 250,    'resume': True,    'resume_from': 'ckpt.pt',    'save_plateau': True,    'resume_lr_scale': 1.0,    'out_dir': '/kaggle/working',    'cache_dir': '/kaggle/temp/t4llm',    'tokenizer_path': '/kaggle/working/tokenizer.json'}POST = {   'n_layer': 12,    'n_head': 12,    'dim': 768,    'mlp_mult': 4,    'seq_len': 1024,    'softcap': 15.0,    'relu2_clamp': 200.0,    'eot_token': '<|endoftext|>',    'base_checkpoint': 'model_final.pt',    'de_ratio': 0.2,    'sft_tokens': 200000000,    'sft_val_blocks': 512,    'micro_bs': 8,    'grad_accum': 8,    'sft_epochs': 2,    'time_budget_h': 4.5,    'lr_muon': 0.0025,    'muon_momentum': 0.95,    'ns_steps': 5,    'muon_weight_decay': 0.0,    'lr_embed': 0.03,    'lr_head': 0.0008,    'lr_scalar': 0.002,    'adam_betas': (0.9, 0.95),    'grad_clip': 1.0,    'warmup_steps': 50,    'final_lr_frac': 0.05,    'min_loss_scale': 64.0,    'max_loss_scale': 65536.0,    'compile': True,    'compile_mode': 'default',    'fp16_allreduce': True,    'seed': 1337,    'nccl_timeout_min': 60,    'log_every': 10,    'val_every': 200,    'save_every': 100,    'sample_tokens': 160,    'sample_prompts': [   'Erklaere einem Kind in drei Saetzen, warum der Himmel blau ist.',                          'Schreibe eine hoefliche Absage auf eine Einladung zu einer '                          'Hochzeit.',                          'What is the difference between weather and climate?',                          'Give me three tips for learning a new language as an adult.'],    'out_dir': '/kaggle/working',    'cache_dir': '/kaggle/temp/t4llm',    'hf_cache_dir': '/kaggle/temp/hf',    'tokenizer_path': '/kaggle/working/tokenizer.json'}

In [ ]:
%%writefile /kaggle/working/dataio.py"""Streaming + filtering + packing of FineWeb-Edu (EN) and German Commons (DE).Design notes------------* We resolve the *explicit parquet file list* of each repo up front and hand a  disjoint slice to every (rank, dataloader-worker) pair. That gives  deterministic, non-overlapping sharding without downloading the same bytes  twice; `datasets`' automatic sharding is version-dependent, this is not.* Column projection is pushed into the parquet reader, so `license`, `url`,  `file_path` etc. never cross the wire.* Every stream is infinite -- files are reshuffled and replayed forever, so the  training loop never has to think about epoch boundaries -- and every network  error retries instead of killing the run.* German Commons files are read through several concurrent file streams per  worker, because one parquet file holds a single source; reading one at a time  would make each batch homogeneous.On `ocr_score`--------------German Commons scores every document 0-100 with OCRoscope. Despite the name itbehaves as a general "is this clean flowing prose" score, not just an OCR check:sampling one parquet file per source gives, token-weighted, 99% kept forpolitical speeches and 90% for YouTube transcripts but only 32% for Wikipediaand 1.4% for Europeana Newspapers. The score is NULL for a small minority ofdocuments (0-17% depending on source, usually <5%); `keep_null_ocr_score`decides what happens to those. Default is False -- strictly `> min_ocr_score`."""from __future__ import annotationsimport itertoolsimport osimport queueimport randomimport threadingimport timeimport numpy as np# Every corpus this project can stream, keyed by the name the config uses.## `columns` is both the pyarrow projection and the declaration of which filters# apply: a spec that lists "score" gets the FineWeb-Edu classifier threshold, one# that lists "ocr_score" gets the German Commons OCR filter, and one that lists# neither is taken as-is. Projection is not a nicety here -- FineWeb2-HQ carries# an `embeddings` column that is most of its 663 GB, and reading it would make the# corpus unusable over a Kaggle connection.SOURCES: dict[str, dict] = {    "fineweb-edu": dict(        repo="HuggingFaceFW/fineweb-edu", prefix="sample/10BT", lang="en",        columns=["text", "score"],        note="10B GPT-2 tokens of classifier-filtered educational web text"),    "cosmopedia": dict(        repo="HuggingFaceTB/smollm-corpus", prefix="cosmopedia-v2/", lang="en",        columns=["text"],        note="~28B tokens of LLM-authored textbooks; the ingredient most credited "             "for SmolLM's quality at small scale"),    "fineweb2-hq": dict(        repo="epfml/FineWeb2-HQ", prefix="deu_Latn/", lang="de",        columns=["text"],        note="top 10% of FineWeb2 German by a knowledge-density classifier; its "             "card reports matching FineWeb2 with 6x fewer tokens, measured on "             "German MMLU at 1B parameters"),    # Kept so the old mixture can still be reproduced for an A/B, not in the    # default mixture: 82% of German Commons is historical newspaper and    # cultural-heritage OCR (News 72.67B + Cultural 54.49B of 154.56B tokens),    # and the modern remainder is dominated by YouTube transcripts.    "german-commons": dict(        repo="coral-nlp/german-commons", prefix="", lang="de",        columns=["text", "ocr_score"], exclude=None,        note="native but 82% pre-1950 OCR; superseded by fineweb2-hq"),}# German Commons sources that are historical (pre-~1950 newspapers, Fraktur# books, 19th century literature). Excluded by default: most of their documents# fail an `ocr_score > 95` filter -- Europeana Newspapers keeps 1.4% of its# tokens, Deutsches Zeitungsportal 6.7% -- so streaming them mostly burns# bandwidth, and what survives is 19th-century German rather than the modern# German we want.GC_HISTORICAL = (    "source=Anno",    "source=Europeana Newspapers",    "source=Deutsches Zeitungsportal",    "source=BLBooks",    "source=GermanPD",    "source=SBB Fulltexts",    "source=DiBiLit",    "source=DiBiPhil",    "source=Reichtagsprotokolle",    "source=Polytechnisches Journal",)_FILE_CACHE: dict[str, list[str]] = {}def _repo_parquet_files(repo: str) -> list[str]:    if repo not in _FILE_CACHE:        from huggingface_hub import HfApi        _FILE_CACHE[repo] = sorted(            f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet"))    return _FILE_CACHE[repo]SOURCES["german-commons"]["exclude"] = GC_HISTORICALdef source_files(name: str, only_sub: list[str] | None = None) -> list[str]:    """Every parquet file of one registry source, as hf:// paths."""    spec = SOURCES[name]    files = [f for f in _repo_parquet_files(spec["repo"]) if f.startswith(spec["prefix"])]    if only_sub:                       # German Commons sub-source selection (diagnose2)        files = [f for f in files if any(f"source={s}" in f for s in only_sub)]    elif spec.get("exclude"):        files = [f for f in files if not any(h in f for h in spec["exclude"])]    if not files:        raise RuntimeError(f"no parquet files under {spec['prefix']!r} in {spec['repo']}")    return [f"hf://datasets/{spec['repo']}/{f}" for f in files]def _spec_for(path: str) -> dict:    """Which registry source a file belongs to, from its hf:// path."""    for spec in SOURCES.values():        if path.startswith(f"hf://datasets/{spec['repo']}/"):            return spec    raise KeyError(f"no source in SOURCES matches {path!r}")def shard_files(files: list[str], index: int, num_shards: int, seed: int = 0) -> list[str]:    """Deterministic disjoint round-robin slice, shuffled first so that a shard    is not one contiguous source."""    files = list(files)    random.Random(seed).shuffle(files)    out = files[index::num_shards]    return out or [files[index % len(files)]]# --------------------------------------------------------------------------- ## document streams# --------------------------------------------------------------------------- ## Read parquet with pyarrow over HfFileSystem rather than `datasets`. Two reasons:#  1. `datasets.IterableDataset` shards itself across torch DataLoader workers.#     With one file handed to one load_dataset() call the shard count is 1, so#     every worker but #0 silently yields nothing -- which turns the retry loop#     into a busy loop and takes the host down with it.#  2. Direct row-group access lets us shuffle row groups, project columns, and#     cap how much we take from one file before moving on.def _malloc_trim():    """Hand freed arenas back to the OS.    glibc keeps freed blocks in per-thread arenas, so RSS after a fill stays at    the high-water mark of the arrow table plus the tokenisation transients. The    loader's RSS plateaued at 8.0 GiB per worker that way, which is what made    four workers exceed the 31 GB host and get SIGKILLed. Freeing the memory in    Python is not enough; it has to be returned.    """    try:        import ctypes        ctypes.CDLL("libc.so.6").malloc_trim(0)    except Exception:  # noqa: BLE001 - not glibc, or no libc: nothing to do        passdef _rss_gb():    """Resident set size of this process, in GiB. Used in the data-loader log:    a worker being SIGKILLed by the OOM killer is otherwise undiagnosable."""    try:        import resource        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20    except Exception:  # noqa: BLE001        return float("nan")_FS = None_FS_PID = Nonedef _fs():    """One HfFileSystem per process. Keyed on the PID because DataLoader workers    are forked, and an inherited fsspec instance shares HTTP connection state    with the parent."""    global _FS, _FS_PID    if _FS is None or _FS_PID != os.getpid():        from huggingface_hub import HfFileSystem        _FS, _FS_PID = HfFileSystem(), os.getpid()    return _FSdef _reset_fs():    global _FS    _FS = Nonedef parquet_rows(paths: list[str], seed: int, log=print,                 max_chars_per_file: int | None = None, slice_rows: int = 512):    """Infinite stream of `(spec, row)`, projected to each file's own columns.    The spec travels with the row because one path list can now mix repos with    different columns and different quality fields, and the consumer has to know    which filters apply.    Files and row groups are both reshuffled; `max_chars_per_file` moves on to a    different file after that much text, which matters because a single German    Commons row group can hold 50k documents -- without a cap one worker would    spend the entire run inside two or three source files.    Peak memory is one row group of the projected columns (~0.9 GB for the big    German Commons files, ~5 MB for FineWeb-Edu).    """    import pyarrow.parquet as pq    rng = random.Random(seed)    paths = list(paths)    while True:        order = paths[:]        rng.shuffle(order)        for path in order:            spec = _spec_for(path)            fs_path = path.replace("hf://", "")            emitted = 0            for attempt in range(5):                try:                    with _fs().open(fs_path, "rb") as fh:                        pf = pq.ParquetFile(fh)                        groups = list(range(pf.num_row_groups))                        rng.shuffle(groups)                        for gi in groups:                            table = pf.read_row_group(gi, columns=spec["columns"])                            idx = list(range(table.num_rows))                            rng.shuffle(idx)                            for lo in range(0, len(idx), slice_rows):                                chunk = table.take(idx[lo:lo + slice_rows]).to_pylist()                                for row in chunk:                                    emitted += len(row.get("text") or "")                                    yield spec, row                                if max_chars_per_file and emitted >= max_chars_per_file:                                    break                            del table                            if max_chars_per_file and emitted >= max_chars_per_file:                                break                    break                except Exception as exc:  # noqa: BLE001 - network flakiness is expected                    log(f"[data] {type(exc).__name__} on {path.rsplit('/', 1)[-1]} "                        f"(attempt {attempt + 1}/5): {str(exc)[:150]}")                    # Drop the cached HfFileSystem. Once its underlying HTTP session                    # has gone bad every subsequent open on it fails the same way,                    # so retrying against the same instance just burns the budget.                    _reset_fs()                    time.sleep(min(30, 3 * 2**attempt))def keep_row(row, spec, min_chars=200, min_edu_score=0.0, min_ocr_score=90,             keep_null_ocr_score=False):    """The text of a row if it passes its source's filters, else None.    Which filters apply is read off the source's `columns`, so a corpus that    carries no quality score -- Cosmopedia, FineWeb2-HQ, both already filtered    upstream -- passes on length alone instead of needing its own branch here.    """    text = row.get("text")    if not text or len(text) < min_chars:        return None    cols = spec["columns"]    if "score" in cols:                       # FineWeb-Edu classifier score        s = row.get("score")        if s is not None and s < min_edu_score:            return None    if "ocr_score" in cols:                   # German Commons, strictly ">"        s = row.get("ocr_score")        if s is None:            if not keep_null_ocr_score:                return None        elif s <= min_ocr_score:            return None    return textdef docs(paths, seed, min_chars=200, min_edu_score=0.0, min_ocr_score=90,         keep_null_ocr_score=False, max_chars_per_file=None, log=print, **_):    """Infinite stream of document texts from any mix of registry sources."""    for spec, row in parquet_rows(paths, seed, log=log,                                  max_chars_per_file=max_chars_per_file):        text = keep_row(row, spec, min_chars, min_edu_score, min_ocr_score,                        keep_null_ocr_score)        if text:            yield textdef weighted_docs(streams, weights):    """Interleave several document streams so each gets its share of *characters*.    The same deficit rule `pack_mixed` uses across languages, applied within one.    A coin flip per document would not do it: Cosmopedia articles are uniform and    short while FineWeb-Edu pages vary by orders of magnitude, so sampling by    document gives a mixture nothing like the configured one.    """    total = sum(weights) or 1.0    share = [w / total for w in weights]    seen = [0] * len(streams)    while True:        n = sum(seen) or 1        k = min(range(len(streams)), key=lambda i: seen[i] / n - share[i])        text = next(streams[k])        seen[k] += len(text)        yield textclass _DeMixer:    """K German sources served concurrently from tokenised buffers.    A German Commons row group is ~1 GB and takes ~40 s to fetch, so sources    cannot be switched per batch at the parquet level. Each stream reads    `chars_per_fill` once, tokenises it (80M chars -> ~23M tokens -> ~46 MB as    uint16) and drops the arrow table; batches then sample across all K buffers.    Reading one source at a time -- the original behaviour, ~1.6h per worker --    is sequential domain training and cost +0.11 to +0.26 nats on sources the    model had moved away from.    One long-lived producer thread per stream, each one buffer ahead via a    depth-1 queue. A synchronous refill blocks the entire rank: the first version    stalled a rank past the NCCL collective timeout and killed an 8h run at step    7950. Spawning a thread per refill instead was worse -- thread explosion, and    a shared random.Random used across threads. HTTP, pyarrow decode and the Rust    tokeniser all release the GIL, so K fixed threads genuinely overlap.    """    ENCODE_CHUNK = 512        # documents per encode_batch call; bounds peak RSS    def __init__(self, paths, tokenizer, eot_id, seed, n_streams, chars_per_fill,                 min_chars, filters, log=print):        self.tok, self.eot, self.log = tokenizer, eot_id, log        self.chars, self.min_chars = chars_per_fill, min_chars        self.filters = filters          # min_edu_score / min_ocr_score / keep_null        n = max(1, min(n_streams, len(paths)))        self.slices = [paths[i::n] for i in range(n)]        self.rng = random.Random(seed)          # main thread only        self.bufs: list[list] = [[] for _ in range(n)]        self.pos = [0] * n        self.queues = [queue.Queue(maxsize=1) for _ in range(n)]        # Only one stream may hold an arrow table at a time. A German row group is        # ~1 GB, so four producers fetching concurrently is ~4 GB per worker and        # ~16 GB across four workers -- which is how the OOM killer took a run.        # The point of the thread is to overlap a fetch with *training*, not with        # three other fetches, and each buffer lasts hours, so serialising the        # download costs nothing.        self.fetch_lock = threading.Lock()        for k in range(n):            threading.Thread(target=self._producer, args=(k, seed + 7919 * k),                             daemon=True).start()    def _producer(self, k, seed):        """Fetch this stream's slices forever; the depth-1 queue paces us."""        rng = random.Random(seed)               # per-thread: Random is not thread-safe        paths = self.slices[k][:]        rng.shuffle(paths)        cursor = 0        while True:            path = paths[cursor % len(paths)]            # The first fill of each stream is deliberately small: four full-size            # fills per worker, serialised, is minutes of dead time at startup.            chars = self.chars // 8 if cursor == 0 else self.chars            cursor += 1            try:                docs = self._fetch(k, path, rng, chars)            except Exception as exc:  # noqa: BLE001                self.log(f"[de] stream {k} fill failed: {type(exc).__name__} {str(exc)[:120]}")                time.sleep(5)                continue            if docs:                self.queues[k].put(docs)        # blocks until the consumer takes it    def _fetch(self, k, path, rng, chars=None):        chars = chars or self.chars        texts, total = [], 0        # Hold the lock only while an arrow table is alive; tokenisation afterwards        # needs no table and may overlap freely across streams.        with self.fetch_lock:            # A fresh generator per fill, closed straight after: leaving it            # suspended would keep its arrow table -- and that GB -- alive.            gen = parquet_rows([path], rng.randrange(1 << 30),                               log=self.log, max_chars_per_file=chars)            try:                for spec, row in gen:                    text = keep_row(row, spec, self.min_chars, **self.filters)                    if not text:                        continue                    texts.append(text)                    total += len(text)                    if total >= chars:                        break            finally:                gen.close()        if not texts:            return []        # Encode in chunks. One encode_batch call over 80M chars materialises        # Encoding objects for ~23M tokens at once -- ids, token strings and masks        # -- which measured 8.0 GiB of peak RSS per worker and got the loader        # OOM-killed twice. Chunking bounds the transient to ENCODE_CHUNK docs;        # the uint16 result is ~47 MB either way.        encode = getattr(self.tok, "encode_batch_fast", self.tok.encode_batch)        docs = []        for i in range(0, len(texts), self.ENCODE_CHUNK):            for e in encode(texts[i:i + self.ENCODE_CHUNK]):                docs.append(np.asarray(e.ids + [self.eot], dtype=np.uint16))            texts[i:i + self.ENCODE_CHUNK] = [""] * len(texts[i:i + self.ENCODE_CHUNK])        del texts        _malloc_trim()        self.log(f"[de] stream {k} <- {source_of(path)} ({total/1e6:.0f}M chars, "                 f"{len(docs)} docs, peak rss {_rss_gb():.1f} GiB)")        return docs    def _left(self, k):        return len(self.bufs[k]) - self.pos[k]    def next_tokens(self):        n = len(self.bufs)        for k in range(n):            if self._left(k) == 0:                try:                    self.bufs[k], self.pos[k] = self.queues[k].get_nowait(), 0                except queue.Empty:                    pass        ready = [k for k in range(n) if self._left(k) > 0]        if not ready:            # Every buffer drained at once -- only at startup in practice. Wait on            # the streams in turn rather than spinning.            while True:                for k in range(n):                    try:                        self.bufs[k], self.pos[k] = self.queues[k].get(timeout=5.0), 0                        ready = [k]                        break                    except queue.Empty:                        continue                if ready:                    break                self.log("[de] all streams still filling ...")        k = self.rng.choice(ready)        self.pos[k] += 1        return self.bufs[k][self.pos[k] - 1]def filters_of(cfg) -> dict:    """The quality thresholds, in the shape `keep_row` and `docs` expect."""    return dict(min_edu_score=cfg["min_edu_score"], min_ocr_score=cfg["min_ocr_score"],                keep_null_ocr_score=cfg["keep_null_ocr_score"])def lang_stream(cfg, paths_by_source, seed, log=print, **kw):    """One document stream over every source of a language, weighted by characters."""    streams, weights = [], []    for i, (name, files) in enumerate(paths_by_source.items()):        streams.append(docs(files, seed + 13 * i, cfg["min_chars"], log=log,                            **filters_of(cfg), **kw))        weights.append(cfg["source_weights"].get(name, 1.0))    return streams[0] if len(streams) == 1 else weighted_docs(streams, weights)def mixed_docs(cfg, seed, log=print):    """Interleave EN/DE so that `de_ratio` holds over *characters*.    Used for the tokenizer corpus, so it sees exactly the training mixture.    A coin flip per document does not produce a 60/40 corpus: document lengths    differ by orders of magnitude between corpora, so `P(document is German)=0.4`    yields something quite different from 40% German text. Taking from whichever    side is behind its share makes the realised mixture match the configured one.    """    en = lang_stream(cfg, cfg["en_files_by_source"], seed, log=log)    de = lang_stream(cfg, cfg["de_files_by_source"], seed + 1, log=log,                     max_chars_per_file=cfg["de_max_chars_per_file"])    de_ratio = cfg["de_ratio"]    n_en = n_de = 0    while True:        if n_de < de_ratio * (n_en + n_de):            text = next(de)            n_de += len(text)        else:            text = next(en)            n_en += len(text)        yield text# --------------------------------------------------------------------------- ## tokenisation + packing# --------------------------------------------------------------------------- #def pack_tokens(doc_iter, tokenizer, seq_len, eot_id, doc_batch=64):    """Tokenise, concatenate with an EOT separator, emit (seq_len+1,) int64    blocks. Consecutive blocks overlap by one token so every position has a    target."""    encode = getattr(tokenizer, "encode_batch_fast", tokenizer.encode_batch)    buf: list[int] = []    need = seq_len + 1    while True:        docs = list(itertools.islice(doc_iter, doc_batch))        if not docs:            return        for enc in encode(docs):            buf.extend(enc.ids)            buf.append(eot_id)        while len(buf) >= need:            yield np.asarray(buf[:need], dtype=np.int64)            del buf[:seq_len]def collect_tokens(doc_iter, tokenizer, n_tokens, eot_id, doc_batch=64):    """Materialise ~n_tokens tokens as one uint16 array (used for the val set)."""    encode = getattr(tokenizer, "encode_batch_fast", tokenizer.encode_batch)    out, total = [], 0    while total < n_tokens:        docs = list(itertools.islice(doc_iter, doc_batch))        if not docs:            break        for enc in encode(docs):            ids = np.asarray(enc.ids + [eot_id], dtype=np.uint16)            out.append(ids)            total += len(ids)    return np.concatenate(out)[:n_tokens] if out else np.zeros(0, dtype=np.uint16)# --------------------------------------------------------------------------- ## torch IterableDataset# --------------------------------------------------------------------------- #def pack_mixed(en_iter, de_mixer, tokenizer, de_ratio, seq_len, eot_id, doc_batch=64,               log=print, report_every=20000):    """EN documents are tokenised on the fly (5 MB row groups, no memory issue);    German comes pre-tokenised out of the mixer. Concatenate with EOT, emit    (seq_len+1,) blocks that overlap by one token.    `de_ratio` is enforced on *tokens*: take from whichever side has emitted less    than its share so far. Choosing the side by coin flip per document -- the    original behaviour -- silently over-weights German, because a German document    is many times longer than a FineWeb-Edu one. `build_val_tokens` already split    the validation set by tokens, so the two disagreed about what "de_ratio=0.4"    meant.    """    encode = getattr(tokenizer, "encode_batch_fast", tokenizer.encode_batch)    buf: list[int] = []    need = seq_len + 1    en_pending: list = []    n_en = n_de = 0    emitted = 0    while True:        if n_de < de_ratio * (n_en + n_de):            ids = de_mixer.next_tokens()      # already EOT-terminated by the mixer            buf.extend(ids.tolist())            n_de += len(ids)        else:            if not en_pending:                docs = list(itertools.islice(en_iter, doc_batch))                if not docs:                    return                en_pending = list(encode(docs))            enc = en_pending.pop()            buf.extend(enc.ids)            buf.append(eot_id)            n_en += len(enc.ids) + 1        while len(buf) >= need:            yield np.asarray(buf[:need], dtype=np.int64)            del buf[:seq_len]            emitted += 1            if report_every and emitted % report_every == 0:                # The realised ratio is the only way to see that the mixture is                # what the config asked for -- it used to be a document ratio, and                # the difference was invisible from the loss.                log(f"[mix] {emitted} blocks | realised German share "                    f"{n_de / max(1, n_en + n_de):.1%} of {(n_en + n_de) / 1e6:.0f}M tokens")def make_train_dataset(cfg, rank, world_size, tokenizer_path):    """The packed training stream for one rank.    `cfg["stream_seed"]` decides *where in the corpora* this stream starts, and    train.py sets it from the step the session resumes at. Without it every    session replays the identical document order: a stream is infinite by    reshuffling and replaying, but it always reshuffles from the same seed, and a    single session gets through well under one parquet file per shard. A    two-session plan therefore trained twice on the first half of the data    instead of once on all of it. The shard *assignment* deliberately stays on    `cfg["seed"]`, or the ranks would stop being disjoint.    """    import torch    class _Packed(torch.utils.data.IterableDataset):        def __iter__(self):            os.environ["TOKENIZERS_PARALLELISM"] = "false"   # one thread per worker process            from tokenizers import Tokenizer            info = torch.utils.data.get_worker_info()            wid = info.id if info else 0            nw = info.num_workers if info else 1            index, shards = rank * nw + wid, world_size * nw            def log(msg):                print(f"[rank{rank}/w{wid}] {msg}", flush=True)            # Shard per source, so every worker sees every corpus rather than one            # worker getting all of Cosmopedia and another all of FineWeb-Edu.            en_by_src = {n: shard_files(f, index, shards, seed=cfg["seed"] + i)                         for i, (n, f) in enumerate(cfg["en_files_by_source"].items())}            de = shard_files([f for fs in cfg["de_files_by_source"].values() for f in fs],                             index, shards, seed=cfg["seed"] + 7)            tok = Tokenizer.from_file(tokenizer_path)            eot = tok.token_to_id(cfg["eot_token"])            resume = 7919 * int(cfg.get("stream_seed", 0))            en_iter = lang_stream(cfg, en_by_src, cfg["seed"] * 1000 + index + resume, log=log)            de_mixer = _DeMixer(de, tok, eot, cfg["seed"] * 977 + index + resume,                                cfg["de_streams"],                                cfg["de_max_chars_per_file"], cfg["min_chars"],                                filters_of(cfg), log=log)            for block in pack_mixed(en_iter, de_mixer, tok, cfg["de_ratio"], cfg["seq_len"],                                    eot, cfg["doc_batch"], log=log):                yield torch.from_numpy(block)    return _Packed()# --------------------------------------------------------------------------- ## balanced validation set# --------------------------------------------------------------------------- #def build_val_tokens(cfg, tokenizer, eot_id, total_tokens, log=print):    """One equally sized bucket per source.    The obvious construction -- take the first N tokens off the mixed stream --    silently produces a single-source metric, because one German Commons row    group holds far more than N tokens. That made the reported val loss track    whichever file came first (Drucksachen des Bundestages, as it happened),    which in turn made the best-checkpoint selection pick the wrong model: a    checkpoint 0.44 nats *worse* on English scored better overall simply because    it had recently trained on Bundestag-like text.    The buckets are one per *registry source* now, with the two languages    weighted by de_ratio, so the metric moves with the mixture rather than with    whichever corpus happens to sit at the front of the file.    """    by_src = cfg["val_files_by_source"]    per_lang = {"en": [n for n in by_src if SOURCES[n]["lang"] == "en"],                "de": [n for n in by_src if SOURCES[n]["lang"] == "de"]}    share = {"en": 1 - cfg["de_ratio"], "de": cfg["de_ratio"]}    buckets = []    for lang, names in per_lang.items():        if not names:            continue        per_src = int(total_tokens * share[lang]) // len(names)        for name in names:            b = collect_tokens(                docs(by_src[name], 7, cfg["min_chars"], log=lambda m: None,                     **filters_of(cfg)),                tokenizer, per_src, eot_id)            log(f"  val {lang.upper()} {name}: {len(b)/1e3:.0f}k tokens")            buckets.append(b)    return np.concatenate(buckets)def source_of(path: str) -> str:    """The finest source name a path carries: German Commons' own `source=` when    present (the `[de] stream k <- X` log line wants it), else the registry name."""    for part in path.split("/"):        if part.startswith("source="):            return part[len("source="):]    for name, spec in SOURCES.items():        if path.startswith(f"hf://datasets/{spec['repo']}/"):            return name    return "?"

In [ ]:
%%writefile /kaggle/working/model.py"""GPT + Muon, tuned for what a Tesla T4 (sm75) can actually do.Architecture follows KellerJordan/modded-nanogpt: RMSNorm without learnablegain, no biases, rotary embeddings, QK-norm, ReLU^2 MLP, value-residuallearning, U-net skip connections between the layer halves, per-blockembedding-shortcut lambdas, zero-initialised output projections and a tanhlogit softcap.Turing deviations:  * bf16 has no tensor-core path on sm75, so Newton-Schulz iterates in fp16    (normalisation still happens in fp32 -- a raw gradient's Frobenius norm    overflows fp16 easily).  * FlashAttention-2 needs sm80+. `F.scaled_dot_product_attention` falls back    to the memory-efficient cutlass kernel here, which is the fastest fused    causal attention available on this GPU."""from __future__ import annotationsimport torchimport torch.distributed as distimport torch.nn as nnimport torch.nn.functional as FNS_DTYPE = torch.float16# --------------------------------------------------------------------------- ## Muon# --------------------------------------------------------------------------- #def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int) -> torch.Tensor:    """Quintic Newton-Schulz iteration -> an approximate orthogonalisation of G.    Batches over leading dimensions, so a (3, d, d) qkv parameter gets q, k and    v orthogonalised independently.    """    assert G.ndim >= 2    a, b, c = 3.4445, -4.7750, 2.0315    X = G.float()    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)    X = X.to(NS_DTYPE)    transposed = G.size(-2) > G.size(-1)    if transposed:        X = X.mT    for _ in range(steps):        A = X @ X.mT        B = b * A + c * (A @ A)        X = a * X + B @ X    if transposed:        X = X.mT    return Xclass _Immediate:    def wait(self):        return Noneclass Muon(torch.optim.Optimizer):    """Momentum SGD whose update is orthogonalised by Newton-Schulz.    The Newton-Schulz work is sharded: rank r only orthogonalises parameters    r, r+W, r+2W, ... and the updates are all-gathered, so each GPU does 1/W of    the iteration and the collective for matrix i overlaps with the iteration    for matrix i+W.    """    def __init__(self, params, lr=0.025, momentum=0.95, nesterov=True, ns_steps=5,                 weight_decay=0.0, rank=0, world_size=1):        self.rank, self.world_size = rank, world_size        params = list(params)        groups = [dict(params=[p for p in params if p.numel() == n], numel=n)                  for n in sorted({p.numel() for p in params})]        super().__init__(groups, dict(lr=lr, momentum=momentum, nesterov=nesterov,                                      ns_steps=ns_steps, weight_decay=weight_decay))        # scratch all-gather buffers, keyed by numel. Deliberately *not* stored in        # param_groups: Optimizer.state_dict() serialises those, and these are        # ~170 MB of pure scratch that has no business in a checkpoint.        self._bufs = {}        for g in self.param_groups:            buf = torch.empty(world_size, g["numel"], dtype=NS_DTYPE, device="cuda")            self._bufs[g["numel"]] = (buf, [buf[i] for i in range(world_size)])    @torch.no_grad()    def step(self):  # noqa: D102        for group in self.param_groups:            params = group["params"]            buffer, views = self._bufs[group["numel"]]            handle, params_world = None, None            def flush():                if params_world is None:                    return                handle.wait()                for p, upd in zip(params_world, views):                    # Decoupled weight decay. Muon's update has a fixed spectral                    # norm by construction, so without decay every weight matrix                    # grows monotonically -- and with it the activations. On bf16                    # that is merely untidy; in fp16 the activations eventually                    # leave the representable range and the run dies.                    if group["weight_decay"]:                        p.mul_(1.0 - group["lr"] * group["weight_decay"])                    # rectangular matrices get a sqrt(fan-out/fan-in) correction                    scale = max(1.0, p.size(-2) / p.size(-1)) ** 0.5                    p.add_(upd.view_as(p).to(p.dtype), alpha=-group["lr"] * scale)            for base in range(0, len(params), self.world_size):                idx = base + self.rank                if idx < len(params):                    p = params[idx]                    state = self.state[p]                    if "momentum_buffer" not in state:                        state["momentum_buffer"] = torch.zeros_like(p.grad)                    buf = state["momentum_buffer"]                    buf.lerp_(p.grad, 1 - group["momentum"])                    g = p.grad.lerp_(buf, group["momentum"]) if group["nesterov"] else buf                    upd = zeropower_via_newtonschulz5(g, group["ns_steps"]).flatten()                else:                    upd = views[self.rank]                flush()                if self.world_size > 1:                    handle = dist.all_gather_into_tensor(buffer, upd, async_op=True)                else:                    buffer[0].copy_(upd)                    handle = _Immediate()                params_world = params[base:base + self.world_size]            flush()# --------------------------------------------------------------------------- ## model# --------------------------------------------------------------------------- #def norm(x: torch.Tensor) -> torch.Tensor:    return F.rms_norm(x, (x.size(-1),))class Rotary(nn.Module):    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10000.0):        super().__init__()        inv = base ** (-torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)        theta = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv)        self.register_buffer("cos", theta.cos(), persistent=False)        self.register_buffer("sin", theta.sin(), persistent=False)    def forward(self, x: torch.Tensor, offset: int = 0) -> torch.Tensor:  # (B, T, H, D)        T = x.size(1)        cos = self.cos[offset:offset + T].view(1, T, 1, -1)        sin = self.sin[offset:offset + T].view(1, T, 1, -1)        x1, x2 = x.float().chunk(2, dim=-1)        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)class Attention(nn.Module):    def __init__(self, dim: int, n_head: int, layer_idx: int = 0):        super().__init__()        self.n_head, self.head_dim = n_head, dim // n_head        self.layer_idx = layer_idx        # (3, dim, dim) rather than one (3*dim, dim) matrix: Newton-Schulz        # batches over leading dims, so q/k/v are orthogonalised separately.        self.qkv_w = nn.Parameter(torch.empty(3, dim, dim).uniform_(-dim**-0.5, dim**-0.5))        self.o_w = nn.Parameter(torch.zeros(dim, dim))   # zero-init output projection        self.lam = nn.Parameter(torch.tensor(0.5))       # value-residual mixing    def forward(self, x, v0, rotary, cache=None, attn_mask=None):        B, T, C = x.shape        qkv = F.linear(x, self.qkv_w.flatten(end_dim=1)).view(B, T, 3, self.n_head, self.head_dim)        q, k, v = qkv.unbind(dim=2)        q, k = norm(q), norm(k)                     # QK-norm        if cache is None:            q, k = rotary(q), rotary(k)        else:            q, k = rotary(q, cache.pos), rotary(k, cache.pos)        if v0 is None:            v0 = v        # Value-residual learning. Applied unconditionally -- in layer 0 it is a        # numeric no-op, but it keeps `lam` in the autograd graph, and DDP        # rejects parameters that never receive a gradient.        v = torch.lerp(v, v0, self.lam)        if cache is None:            y = F.scaled_dot_product_attention(                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True)        else:            # The cache holds the *post-lerp* v, which is what SDPA consumed when            # the position was first written -- recomputing the value residual for            # a cached position is impossible, since v0 of that step is gone.            k, v = cache.append(self.layer_idx, k, v)            y = F.scaled_dot_product_attention(                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),                attn_mask=attn_mask, is_causal=attn_mask is None and q.size(1) == k.size(1))        y = y.transpose(1, 2).reshape(B, T, C)        return F.linear(y, self.o_w), v0class MLP(nn.Module):    def __init__(self, dim: int, mult: int = 4, relu2_clamp: float = 128.0):        super().__init__()        hidden = mult * dim        self.relu2_clamp = relu2_clamp        self.up = nn.Parameter(torch.empty(hidden, dim).uniform_(-dim**-0.5, dim**-0.5))        self.down = nn.Parameter(torch.zeros(dim, hidden))   # zero-init    def forward(self, x):        # NOTE: nothing that syncs (`.item()`) or reads a mutable Python attribute        # belongs in here. An earlier version measured clamp saturation inline and        # cost 25% throughput and +4 GiB: the sync broke the compiled graph at all        # 12 MLPs, killing inductor's fusion, and the attribute read made dynamo        # recompile every time the flag flipped. Weight growth is tracked in the        # training loop instead, where it is free; use diagnose.py for the        # activation distribution itself.        h = F.relu(F.linear(x, self.up))        # ReLU^2 is the only op here that doubles the exponent, and fp16 tops out        # at 65504 -- relu(h)=256 already overflows. bf16 (what modded-nanogpt        # runs) has fp32 range and never sees this; on sm75 an unclamped square        # is a live failure mode that GradScaler cannot repair, because the        # overflow is in the activations, not in the gradient magnitude. The        # scaler backs off anyway, the scale collapses, 1/scale becomes inf, and        # every gradient is NaN from then on. 128^2 = 16384 leaves 4x headroom        # and sits well above any activation healthy training produces.        return F.linear(h.clamp(max=self.relu2_clamp).square(), self.down)   # ReLU^2class Block(nn.Module):    def __init__(self, dim: int, n_head: int, mlp_mult: int, relu2_clamp: float = 128.0,                 layer_idx: int = 0):        super().__init__()        self.attn = Attention(dim, n_head, layer_idx)        self.mlp = MLP(dim, mlp_mult, relu2_clamp)        self.lam = nn.Parameter(torch.tensor([1.0, 0.0]))   # residual / embedding-shortcut mix    def forward(self, x, v0, x0, rotary, cache=None, attn_mask=None):        x = self.lam[0] * x + self.lam[1] * x0        h, v0 = self.attn(norm(x), v0, rotary, cache, attn_mask)        x = x + h        x = x + self.mlp(norm(x))        return x, v0class GPT(nn.Module):    def __init__(self, vocab_size, n_layer, n_head, dim, max_seq_len, mlp_mult=4, softcap=15.0,                 relu2_clamp=128.0):        super().__init__()        self.softcap = softcap        self.max_seq_len = max_seq_len        self.embed = nn.Embedding(vocab_size, dim)        self.embed.weight.data.uniform_(-0.5 * dim**-0.5, 0.5 * dim**-0.5)        self.rotary = Rotary(dim // n_head, max_seq_len)        self.blocks = nn.ModuleList(            [Block(dim, n_head, mlp_mult, relu2_clamp, i) for i in range(n_layer)])        self.lm_head = nn.Linear(dim, vocab_size, bias=False)        self.lm_head.weight.data.zero_()   # start from a uniform predictive distribution        self.n_skip = n_layer // 2        self.skip_w = nn.Parameter(torch.ones(self.n_skip))    def body(self, idx, cache=None, attn_mask=None):        """The residual stream after the final norm -- what the lm_head reads.        Split out because the reward model scores this tensor instead of        predicting a token from it; the training path is unchanged.        """        x = norm(self.embed(idx))        x0, v0 = x, None        skips = []        for i, block in enumerate(self.blocks):            if i >= self.n_skip:                       # U-net decoder half                x = x + self.skip_w[i - self.n_skip] * skips.pop()            x, v0 = block(x, v0, x0, self.rotary, cache, attn_mask)            if i < self.n_skip:                        # U-net encoder half                skips.append(x)        return norm(x)    def forward(self, idx, targets=None, cache=None, attn_mask=None, last_only=False):        """`targets` may contain -100 at positions that should not be trained on        (F.cross_entropy ignores those and leaves them out of the mean), which is        how SFT masks the prompt."""        x = self.body(idx, cache, attn_mask)        if cache is not None:            cache.advance(idx.size(1))        if last_only:            x = x[:, -1:]        logits = self.lm_head(x)        logits = self.softcap * torch.tanh(logits.float() / self.softcap)        if targets is None:            return logits        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))    def param_groups(self):        hidden = [p for p in self.blocks.parameters() if p.ndim >= 2]        scalar = [p for p in self.parameters() if p.ndim < 2]        return hidden, scalar, [self.embed.weight], [self.lm_head.weight]# --------------------------------------------------------------------------- ## incremental decoding# --------------------------------------------------------------------------- #class KVCache:    """Pre-allocated per-layer key/value cache.    Without one, sampling n tokens re-runs the whole prefix n times -- O(n^2) in    a model this small is entirely decode overhead, and it makes GRPO rollouts    (thousands of sampled tokens per optimizer step) the dominant cost of the    run. Allocated once rather than grown with torch.cat, which would otherwise    copy the entire cache on every decoded token.    Sized (n_layer, batch, max_len, n_head, head_dim): at the default rollout    shape (12, 32, 768, 12, 64) that is 2 x 216 MiB in fp16.    The dtype is taken from the first key written rather than fixed, because the    compute dtype is whatever autocast decided -- pinning it to fp16 both silently    rounds an fp32 forward and hands SDPA a key whose dtype does not match the    query, which is an error rather than a wrong answer. Allocation is therefore    deferred to the first write.    """    def __init__(self, n_layer, batch, max_len, n_head, head_dim, device, dtype=None):        self.shape = (n_layer, batch, max_len, n_head, head_dim)        self.device, self.dtype = device, dtype        self.k = self.v = None        self.max_len, self.batch, self.pos = max_len, batch, 0    def reset(self):        self.pos = 0    def append(self, layer, k, v):        if self.k is None:            self.k = torch.zeros(self.shape, device=self.device, dtype=self.dtype or k.dtype)            self.v = torch.zeros_like(self.k)        b, end = k.size(0), self.pos + k.size(1)        # Sliced to the live batch, not to the allocation: a cache kept across        # rollouts is sized for the largest batch and a smaller one must still work.        self.k[layer, :b, self.pos:end] = k        self.v[layer, :b, self.pos:end] = v        # `.to` is a no-op when the dtypes already agree, which is the normal path.        return self.k[layer, :b, :end].to(k.dtype), self.v[layer, :b, :end].to(v.dtype)    def advance(self, n):        self.pos += ndef causal_pad_mask(q_len, kv_len, valid):    """(B, 1, q_len, kv_len) boolean SDPA mask, True where attention is allowed.    `valid` is (B, kv_len), False on left-padding. Queries are taken to be the    last `q_len` positions of the window, which covers both prefill    (q_len == kv_len) and single-token decode (q_len == 1).    Every position is additionally allowed to attend to itself: a padded query    would otherwise have all of its keys masked, and softmax over an all-`-inf`    row is NaN. Those rows are discarded either way, but a NaN that only appears    with padded batches is a miserable thing to track down.    """    dev = valid.device    pos_k = torch.arange(kv_len, device=dev)    pos_q = torch.arange(kv_len - q_len, kv_len, device=dev)    m = (pos_k[None, :] <= pos_q[:, None])[None] & valid[:, None, :]    return (m | (pos_q[None, :, None] == pos_k[None, None, :])).unsqueeze(1)def _block_repeat_ngrams(logits, recent, recent_valid, n):    """-inf on every token that would complete an n-gram already in the window.    The repetition penalty only tilts the odds and a 125M model walks straight    past it: on the prompts it cannot do, the first run's samples came back with    38-41% of their 4-grams repeated, in loops like "Der Name des Benutzers, der    die Datenbank abruft" five times over. Forbidding the exact continuation is    what breaks the cycle. Done in Python because sampling batches are a handful    of sequences; it never runs during training.    """    for b in range(logits.size(0)):        seq = recent[b][recent_valid[b]].tolist()        if len(seq) < n:            continue        tail = seq[-(n - 1):]        banned = [seq[i + n - 1] for i in range(len(seq) - n + 1)                  if seq[i:i + n - 1] == tail]        if banned:            logits[b, banned] = -float("inf")def _filter_and_sample(logits, recent, recent_valid, temperature, top_k, top_p,                       repetition_penalty, real_vocab, greedy, no_repeat_ngram=0):    """logits: (B, V) fp32, modified in place. Returns (B, 1) sampled ids."""    if real_vocab is not None:        logits[:, real_vocab:] = -float("inf")     # never emit padded vocab slots    if repetition_penalty and repetition_penalty != 1.0 and recent is not None:        prev = logits.gather(1, recent)        # divide positive logits, multiply negative ones -- both push down        pen = torch.where(prev > 0, prev / repetition_penalty, prev * repetition_penalty)        # ...but only for real history. The left-padding slots all carry `pad_id`,        # and penalising it would suppress a genuine token for no reason.        logits.scatter_(1, recent, torch.where(recent_valid, pen, prev))    if no_repeat_ngram > 1 and recent is not None:        _block_repeat_ngrams(logits, recent, recent_valid, no_repeat_ngram)    if greedy or temperature <= 0:        return logits.argmax(-1, keepdim=True)    logits = logits / temperature    if top_k:        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))        logits = logits.masked_fill(logits < v[:, [-1]], -float("inf"))    if top_p and top_p < 1.0:        srt, idx = torch.sort(logits, descending=True, dim=-1)        prob = torch.softmax(srt, dim=-1)        drop = prob.cumsum(dim=-1) - prob > top_p   # keep the first token over the mass        logits = logits.masked_fill(drop.scatter(1, idx, drop), -float("inf"))    return torch.multinomial(F.softmax(logits, dim=-1), 1)@torch.no_grad()def generate_batch(model, prompts, max_new_tokens, eot_id, pad_id=0, temperature=0.85,                   top_k=50, top_p=0.92, repetition_penalty=1.15, repetition_window=128,                   real_vocab=None, device="cuda", greedy=False, cache=None,                   stop_ids=(), no_repeat_ngram=0):    """Sample a continuation for each of `prompts` (a list of token-id lists).    Prompts are **left-padded** so that every sequence's last real token sits at    the same index and one decode step advances all of them at once. Rotary is    relative, so the shifted absolute positions are harmless as long as each    sequence's own tokens stay contiguous; the padding is masked out of the keys.    Returns `(tokens, gen_mask)`. `tokens` is (B, plen + n_new); `gen_mask` is    True exactly at the positions this call sampled, and goes False for a    sequence once it has emitted a stop token -- so a caller can compute a loss    or a reward over the completion only.    """    was_training = model.training    model.eval()    stop = {eot_id, *stop_ids}    B = len(prompts)    plen = max(len(p) for p in prompts)    ids = torch.full((B, plen), pad_id, dtype=torch.long, device=device)    valid = torch.zeros(B, plen, dtype=torch.bool, device=device)    for i, pr in enumerate(prompts):        ids[i, plen - len(pr):] = torch.tensor(pr, dtype=torch.long, device=device)        valid[i, plen - len(pr):] = True    total = plen + max_new_tokens    if cache is None:        att = model.blocks[0].attn        cache = KVCache(len(model.blocks), B, total, att.n_head, att.head_dim, device)    else:        assert cache.batch >= B and cache.max_len >= total, "cache too small for this batch"        cache.reset()    with torch.autocast("cuda", dtype=torch.float16):        logits = model(ids, cache=cache, attn_mask=causal_pad_mask(plen, plen, valid),                       last_only=True)[:, -1].float()    alive = torch.ones(B, dtype=torch.bool, device=device)    gen_cols = []    for t in range(max_new_tokens):        if repetition_penalty != 1.0 or no_repeat_ngram > 1:            recent, recent_valid = ids[:, -repetition_window:], valid[:, -repetition_window:]        else:            recent = recent_valid = None        nxt = _filter_and_sample(logits, recent, recent_valid, temperature, top_k, top_p,                                 repetition_penalty, real_vocab, greedy, no_repeat_ngram)        nxt = torch.where(alive[:, None], nxt, torch.full_like(nxt, pad_id))        gen_cols.append(alive.clone())        ids = torch.cat([ids, nxt], dim=1)        for sid in stop:            alive &= nxt.squeeze(1) != sid        if t == max_new_tokens - 1 or not bool(alive.any()):            break                      # nothing would read the next forward's logits        valid = torch.cat([valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)        with torch.autocast("cuda", dtype=torch.float16):            logits = model(nxt, cache=cache,                           attn_mask=causal_pad_mask(1, cache.pos + 1, valid),                           last_only=True)[:, -1].float()    gen_mask = torch.zeros_like(ids, dtype=torch.bool)    if gen_cols:        gen_mask[:, plen:] = torch.stack(gen_cols, dim=1)    model.train(was_training)    return ids, gen_mask@torch.no_grad()def generate(model, tok, prompt, max_new_tokens=120, temperature=0.85, top_k=50,             top_p=0.92, repetition_penalty=1.15, repetition_window=128,             device="cuda", real_vocab=None, eot_id=None, no_repeat_ngram=0):    """Single-prompt convenience wrapper around `generate_batch`.    An undertrained model of this size loops readily -- plain top-k at T=0.8    produced "die Energie in der Nahe der Oberflache ist, die Energie dann in der    Nahe der Oberflache ist". Most of that is the model, but top-k alone keeps a    long flat tail in play and re-picks the same continuation; nucleus sampling    plus a penalty on recently emitted tokens removes the avoidable part.    """    ids, _ = generate_batch(model, [tok.encode(prompt).ids], max_new_tokens,                            eot_id if eot_id is not None else -1,                            temperature=temperature, top_k=top_k, top_p=top_p,                            repetition_penalty=repetition_penalty,                            repetition_window=repetition_window, real_vocab=real_vocab,                            device=device, no_repeat_ngram=no_repeat_ngram)    return tok.decode(ids[0].tolist())def pack_completions(prompts, ids, gen_mask, pad_id, device="cuda"):    """Left-padded generation output -> right-padded training batch.    Returns `(tokens, completion_mask, last_index)`. `generate_batch` left-pads so    that one decode step advances every sequence, but nothing downstream wants    that layout: the training forward runs plain causal attention with no mask, so    a left pad would sit *before* the real tokens and be attended to, and the    reward model reads its score at an explicit last-token index.    """    plen = max(len(p) for p in prompts)    ids_l, mask_l = ids.tolist(), gen_mask.tolist()    rows, comp_len = [], []    for i, prompt in enumerate(prompts):        comp = [t for t, m in zip(ids_l[i][plen:], mask_l[i][plen:]) if m]        rows.append(list(prompt) + comp)        comp_len.append(len(comp))    width = max(len(r) for r in rows)    tok = torch.full((len(rows), width), pad_id, dtype=torch.long)    mask = torch.zeros(len(rows), width, dtype=torch.bool)    last = torch.zeros(len(rows), dtype=torch.long)    for i, r in enumerate(rows):        tok[i, :len(r)] = torch.tensor(r, dtype=torch.long)        mask[i, len(r) - comp_len[i]:len(r)] = True        last[i] = len(r) - 1    return tok.to(device), mask.to(device), last.to(device)def token_logprobs(model, tokens):    """log p(tokens[:, t] | tokens[:, :t]) for t >= 1 -> (B, T-1).    Uses `cross_entropy(reduction="none")` rather than log_softmax + gather:    both materialise the (B*T, V) fp32 logits, but cross_entropy's backward    recomputes the softmax from them instead of keeping a second copy alive.    """    logits = model(tokens[:, :-1])                    # (B, T-1, V) fp32, softcapped    tgt = tokens[:, 1:]    return -F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),                            reduction="none").view(tgt.shape)# --------------------------------------------------------------------------- ## reward model# --------------------------------------------------------------------------- #class RewardModel(nn.Module):    """A GPT body with a scalar head instead of a token head.    Initialised from the SFT checkpoint and trained with the Bradley-Terry    pairwise loss, so the score is only meaningful up to an additive constant --    which is exactly what GRPO needs, since it normalises rewards within a group    of samples and the constant cancels.    The score is read at the sequence's last real token. `lm_head` is left in    place so the checkpoint loads unchanged; `score_parameters()` is what the    optimizer should actually see.    """    def __init__(self, gpt: "GPT"):        super().__init__()        self.gpt = gpt        self.score = nn.Linear(gpt.embed.weight.size(1), 1, bias=False)        self.score.weight.data.zero_()   # start every sequence at reward 0    def forward(self, idx, last_pos):        h = self.gpt.body(idx)                                    # (B, T, C)        r = self.score(h).squeeze(-1)                             # (B, T)        return r.gather(1, last_pos[:, None]).squeeze(1).float()  # (B,)    def trainable(self):        """Everything except the unused lm_head, split the way the optimizers want."""        hidden = [p for p in self.gpt.blocks.parameters() if p.ndim >= 2]        scalar = [p for p in self.gpt.parameters() if p.ndim < 2]        return hidden, scalar, [self.gpt.embed.weight], [self.score.weight]

In [ ]:
%%writefile /kaggle/working/tokenizer_train.py"""Train a bilingual (EN/DE) byte-level BPE tokenizer on the same mixture themodel will be pretrained on.Why train our own instead of reusing GPT-2's: GPT-2's BPE was fit on EnglishWebText and lands around 3.0 chars/token on German. Fitting 32k merges on theactual 60/40 EN/DE training mixture buys back a chunk of that -- `report()`prints the measured figure for both languages, which is what decides how muchtext a fixed token budget actually covers. The smaller vocab also frees ~13Membedding parameters relative to GPT-2's 50257 entries, and those go into thetransformer body instead.Pre-tokenization uses the GPT-4 (cl100k) split regex plus a byte-levelalphabet, so digits are grouped in <=3s, whitespace runs stay attached, and noinput is ever unrepresentable."""from __future__ import annotationsimport osimport timeimport dataio# GPT-4 / cl100k_base pre-tokenizer pattern (fancy-regex supports the# possessive quantifiers, which is what `tokenizers` uses under the hood).SPLIT_PATTERN = (    r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}"""    r"""| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""")# Reserved during pretraining so that post-training does not have to grow the# embedding matrix. Adding them later works (`chatdata.ensure_chat_tokens` does# exactly that, for tokenizers produced before this existed) but it pushes the# vocabulary past the 128-aligned padding and forces a resize of both the# embedding and the head.CHAT_TOKENS = ("<|im_start|>", "<|im_end|>")def build_tokenizer():    from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers    tok = Tokenizer(models.BPE(unk_token=None, byte_fallback=False))    tok.pre_tokenizer = pre_tokenizers.Sequence([        pre_tokenizers.Split(Regex(SPLIT_PATTERN), behavior="isolated", invert=False),        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),    ])    tok.decoder = decoders.ByteLevel()    return tokdef _corpus(cfg, max_chars, log=print):    docs = dataio.mixed_docs(cfg, seed=cfg["seed"] + 99, log=lambda m: None)    total, t0, nxt = 0, time.time(), 50 << 20    for text in docs:        yield text        total += len(text)        if total >= nxt:            log(f"    corpus {total/2**20:7.0f} MiB  ({total/2**20/(time.time()-t0):5.1f} MiB/s)")            nxt += 50 << 20        if total >= max_chars:            break    log(f"    corpus complete: {total/2**20:.0f} MiB in {time.time()-t0:.0f}s")def train(cfg, out_path: str, log=print):    """Train and save the tokenizer; returns the loaded Tokenizer."""    from tokenizers import Tokenizer, pre_tokenizers, trainers    if os.path.exists(out_path) and not cfg.get("force_retrain_tokenizer"):        log(f"reusing cached tokenizer at {out_path}")        return Tokenizer.from_file(out_path)    tok = build_tokenizer()    trainer = trainers.BpeTrainer(        vocab_size=cfg["vocab_size"],        special_tokens=[cfg["eot_token"], *CHAT_TOKENS],        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),        min_frequency=2,        show_progress=True,    )    t0 = time.time()    log(f"training {cfg['vocab_size']} BPE merges on <= {cfg['tok_train_chars']/2**20:.0f} MiB "        f"of {int((1-cfg['de_ratio'])*100)}/{int(cfg['de_ratio']*100)} EN/DE text ...")    tok.train_from_iterator(_corpus(cfg, cfg["tok_train_chars"], log=log), trainer=trainer)    tok.save(out_path)    log(f"tokenizer trained in {time.time()-t0:.0f}s -> {out_path}")    return Tokenizer.from_file(out_path)def report(tok, cfg, n_docs=200, log=print) -> dict:    """chars/token per source and per language on held-out text.    The number that decides how much text a fixed token budget actually buys --    and, returned here, the constant that makes losses comparable across runs    with different tokenizers: `bits/char = loss / ln(2) / chars_per_token`.    """    import itertools    per_lang: dict[str, list[int]] = {}    for name, files in cfg["val_files_by_source"].items():        lang = dataio.SOURCES[name]["lang"]        gen = dataio.docs(files, 8, cfg["min_chars"], log=lambda m: None,                          **dataio.filters_of(cfg))        # Some corpora hold single documents of several hundred kB (whole Bundestag        # protocols); truncate so the ratio is measured on a spread of documents        # rather than on one outlier.        docs = [d[:20_000] for d in itertools.islice(gen, n_docs)]        if not docs:            continue        enc = tok.encode_batch(docs)        nch = sum(len(d) for d in docs)        ntk = sum(len(e.ids) for e in enc)        log(f"  {lang.upper()} {name:20s} {nch/ntk:5.2f} chars/token   "            f"({ntk/1e3:.0f}k tokens from {nch/1e3:.0f}k chars)")        per_lang.setdefault(lang, [0, 0])        per_lang[lang][0] += nch        per_lang[lang][1] += ntk    out = {lang: ch / tk for lang, (ch, tk) in per_lang.items() if tk}    if "en" in out and "de" in out:        # weighted by the training mixture: this is the divisor for bits/char        out["mixed"] = (1 - cfg["de_ratio"]) * out["en"] + cfg["de_ratio"] * out["de"]        log(f"  {'mixture':24s} {out['mixed']:5.2f} chars/token   "            f"(at de_ratio={cfg['de_ratio']}, used for bits/char)")    for sample in ("Die Bundesregierung hat beschlossen, dass Künstliche Intelligenz "                   "in Schulen eingesetzt wird.",                   "The mitochondrion is the powerhouse of the cell, producing ATP "                   "through oxidative phosphorylation."):        log(f"  {sample[:28]}... -> {tok.encode(sample).tokens}")    return out

In [ ]:
%%writefile /kaggle/working/train.py"""125M bilingual (EN/DE) GPT pretraining on 2x Tesla T4.Launch:  torchrun --standalone --nproc_per_node=2 train.pySpeed work carried over from KellerJordan/modded-nanogpt that survives on Turing:  * Muon on every hidden matrix, sharded across both GPUs.  * ReLU^2 MLP, RMSNorm without gain, no biases, QK-norm, rotary, value    residual, U-net skips, zero-init output projections, tanh logit softcap.  * torch.compile over the whole forward+loss.  * Trapezoidal LR schedule, Muon momentum warmup.  * Short-sequence warmup: the first phase runs at seq_len/2 with twice as many    rows, so tokens/step is unchanged while attention costs half.  * fp16 gradient all-reduce (DDP comm hook), grad-accumulation under no_sync,    gradient buckets as views.Turing-specific: fp16 + GradScaler instead of bf16, SDPA memory-efficientkernel instead of FlashAttention-2, no fp8, no FlexAttention block masks."""from __future__ import annotationsimport contextlibimport datetimeimport gcimport hashlibimport jsonimport mathimport osimport sysimport timeimport numpy as npimport torchimport torch.distributed as distfrom torch.nn.parallel import DistributedDataParallel as DDPsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))import dataio  # noqa: E402from model import NS_DTYPE, GPT, Muon  # noqa: E402CFG = json.load(open(os.environ.get("TRAIN_CONFIG", "config.json")))# --------------------------------------------------------------------------- ## distributed / backend setup# --------------------------------------------------------------------------- #RANK = int(os.environ.get("RANK", 0))LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))WORLD = int(os.environ.get("WORLD_SIZE", 1))MASTER = RANK == 0torch.cuda.set_device(LOCAL_RANK)DEV = torch.device("cuda", LOCAL_RANK)if WORLD > 1:    # The default 10 min collective timeout is shorter than a worst-case data    # stall: a German row group is ~1 GB, and a slow fetch plus retries can block    # a rank for longer. When that happened the other rank sat in an allreduce    # until NCCL aborted the process and took an 8h run with it. A stall should    # cost throughput, not the job.    dist.init_process_group("nccl", device_id=DEV,                            timeout=datetime.timedelta(minutes=CFG["nccl_timeout_min"]))def log(msg):    if MASTER:        print(msg, flush=True)def agree(*vals):    """Rank 0's values, on every rank. Every decision taken from the wall clock goes    through here -- when to stop, how many steps the plan holds.    Each process reads its own clock and starts it at its own moment: seconds apart    after both ranks load a 1.2 GB checkpoint off the same disk. So `time.time() >    deadline` can be true on one rank and false on the other for the same step, and    `fit_total_steps` can round to totals one step apart. Either way one rank leaves    the loop a step early, its evaluation all-reduce pairs with the other rank's    gradient buckets, and NCCL hangs for nccl_timeout_min -- then the run dies before    writing model_final.pt, and the Kaggle session idles to its 12h cap on the quota.    Costs one 24-byte broadcast per step, on a step that already synchronises.    """    if WORLD == 1:        return [float(v) for v in vals]    t = torch.tensor([float(v) for v in vals], dtype=torch.float64, device=DEV)    dist.broadcast(t, src=0)    return t.tolist()torch.manual_seed(CFG["seed"] + RANK)torch.backends.cudnn.benchmark = Truetorch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = Trueimport torch._dynamo  # noqa: E402torch._dynamo.config.cache_size_limit = 64# Exactly two input shapes ever reach the model -- (2B, T/2) during the# short-sequence warmup and (B, T) after. Specialise on both rather than letting# dynamo generalise to a dynamic batch dim, which produces worse kernels.torch._dynamo.config.automatic_dynamic_shapes = Falsetry:    import torch._inductor.config as inductor_config    inductor_config.fx_graph_cache = True    if CFG["coordinate_descent_tuning"]:        inductor_config.coordinate_descent_tuning = Trueexcept Exception:  # noqa: BLE001    pass# --------------------------------------------------------------------------- ## model + optimizers# --------------------------------------------------------------------------- #from tokenizers import Tokenizer  # noqa: E402tokenizer = Tokenizer.from_file(CFG["tokenizer_path"])REAL_VOCAB = tokenizer.get_vocab_size()PAD_VOCAB = (REAL_VOCAB + 127) // 128 * 128   # keep the head matmul tile-alignedEOT = tokenizer.token_to_id(CFG["eot_token"])raw_model = GPT(PAD_VOCAB, CFG["n_layer"], CFG["n_head"], CFG["dim"], CFG["seq_len"],                CFG["mlp_mult"], CFG["softcap"], CFG["relu2_clamp"]).to(DEV)n_params = sum(p.numel() for p in raw_model.parameters())n_emb = raw_model.embed.weight.numel() + raw_model.lm_head.weight.numel()log(f"vocab {REAL_VOCAB} -> padded {PAD_VOCAB}")log(f"params: {n_params/1e6:.1f}M total | {(n_params-n_emb)/1e6:.1f}M non-embedding | "    f"{(n_params-n_emb/2)/1e6:.1f}M in the GPT-2 (tied) counting convention")hidden_p, scalar_p, embed_p, head_p = raw_model.param_groups()assert len(hidden_p) + len(scalar_p) + 2 == len(list(raw_model.parameters()))log(f"muon matrices: {len(hidden_p)} | adam: {len(embed_p)} embed, {len(head_p)} head, "    f"{len(scalar_p)} scalars")opt_adam = torch.optim.Adam(    [dict(params=embed_p, lr=CFG["lr_embed"]),     dict(params=head_p, lr=CFG["lr_head"]),     dict(params=scalar_p, lr=CFG["lr_scalar"])],    betas=tuple(CFG["adam_betas"]), eps=1e-10, fused=True)opt_muon = Muon(hidden_p, lr=CFG["lr_muon"], momentum=CFG["muon_momentum"],                ns_steps=CFG["ns_steps"], weight_decay=CFG["muon_weight_decay"],                rank=RANK, world_size=WORLD)optimizers = [opt_adam, opt_muon]base_lrs = [[g["lr"] for g in o.param_groups] for o in optimizers]all_params = list(raw_model.parameters())# Muon's update has a fixed spectral norm, so weight norms grow monotonically# without decay (measured: 14-19x over 6000 steps). Tracked from the weights# themselves -- a probe inside the forward pass cost 25% throughput._n0 = torch.stack([p.detach().float().norm() for p in hidden_p])_grow_mask = _n0 > 0      # o_w and MLP.down are zero-initialised; ratios would be infinit_norms = _n0[_grow_mask]model = raw_modelif CFG["compile"]:    model = torch.compile(model, mode=CFG["compile_mode"])    log(f"torch.compile enabled (mode={CFG['compile_mode']}); graphs build on the first step")if WORLD > 1:    model = DDP(model, device_ids=[LOCAL_RANK], gradient_as_bucket_view=True,                broadcast_buffers=False, bucket_cap_mb=40)    if CFG["fp16_allreduce"]:        from torch.distributed.algorithms.ddp_comm_hooks import default_hooks        model.register_comm_hook(None, default_hooks.fp16_compress_hook)        log("DDP gradient all-reduce compressed to fp16")scaler = torch.amp.GradScaler("cuda", init_scale=2.0**14, growth_interval=500)def clamp_loss_scale():    """Keep the loss scale inside a sane window.    Two failure modes this prevents. Too high (it grew to 2**18 in testing) and    marginal fp16 gradient overflows become routine. Too low and gradients    underflow to zero in fp16 -- which is far worse with Muon than with Adam,    because Newton-Schulz normalises whatever direction it is given, so a    gradient that is pure quantisation noise still produces a full-magnitude    orthogonal update. Below ~2**-24 the scale is small enough that 1/scale    overflows to inf and every gradient becomes NaN permanently.    """    cur = scaler.get_scale()    lo, hi = CFG["min_loss_scale"], CFG["max_loss_scale"]    if not lo <= cur <= hi:        scaler.update(float(min(max(cur, lo), hi)))        return cur < lo          # True == we are pinned at the floor, i.e. in trouble    return False# --------------------------------------------------------------------------- ## data# --------------------------------------------------------------------------- #data_cfg = {k: CFG[k] for k in            ("en_files_by_source", "de_files_by_source", "source_weights",             "de_ratio", "seed", "min_chars", "min_edu_score",             "min_ocr_score", "keep_null_ocr_score", "de_max_chars_per_file", "de_streams",             "seq_len", "doc_batch",             "eot_token")}# The annealing phase narrows the *mixture*, not a file list: it drops the# broadest source on each side and keeps the densest ones, resolved by the# notebook into these two dicts.ANNEAL = dict(en_files_by_source=CFG["anneal_en_files_by_source"],              de_files_by_source=CFG["anneal_de_files_by_source"],              source_weights=CFG["anneal_source_weights"],              min_edu_score=CFG["anneal_min_edu_score"])def make_loader(overrides=None):    return torch.utils.data.DataLoader(        dataio.make_train_dataset({**data_cfg, **(overrides or {})}, RANK, WORLD,                                  CFG["tokenizer_path"]),        batch_size=CFG["micro_bs"], num_workers=CFG["data_workers"], pin_memory=True,        drop_last=True, persistent_workers=CFG["data_workers"] > 0,        prefetch_factor=CFG["prefetch_factor"] if CFG["data_workers"] > 0 else None)loader = train_iter = None   # built after resume, once the schedule is knowndef next_batch():    global train_iter    try:        return next(train_iter)    except StopIteration:        train_iter = iter(loader)        return next(train_iter)os.makedirs(CFG["cache_dir"], exist_ok=True)if MASTER:    os.makedirs(CFG["out_dir"], exist_ok=True)# In out_dir, not cache_dir: /kaggle/temp is wiped between sessions, so a# multi-session plan re-downloaded and re-tokenised the held-out files every# time -- 4 MB of cache for several minutes of a session that has none to spare.# Keyed on the *tokenizer*, not just its vocab size: a rerun with a new corpus# trains a new tokenizer of the same size, and reusing the old token ids would# silently score the model against a validation set encoded by a different BPE._tok_key = hashlib.sha1(open(CFG["tokenizer_path"], "rb").read()).hexdigest()[:10]VAL_PATH = os.path.join(CFG["out_dir"], f"val_bal_{CFG['val_tokens']}_{_tok_key}.npy")if not os.path.exists(VAL_PATH):    # Build on rank 0 only. Both ranks used to build it independently, which    # downloaded and tokenised the held-out files twice; and `np.save` is not    # atomic, so the loser of the race could read a half-written file. The    # barrier is covered by nccl_timeout_min, which is why that has to be > 10 min.    if MASTER:        log("building held-out validation set (files disjoint from training) ...")        vt = dataio.build_val_tokens(CFG, tokenizer, EOT, CFG["val_tokens"], log=log)        np.save(VAL_PATH + ".tmp.npy", vt)        os.replace(VAL_PATH + ".tmp.npy", VAL_PATH)        del vt    if WORLD > 1:        dist.barrier()val_tokens = np.load(VAL_PATH)n_val = (len(val_tokens) - 1) // CFG["seq_len"]_v = val_tokens.astype(np.int64)val_x = torch.from_numpy(_v[: n_val * CFG["seq_len"]]).view(n_val, -1).pin_memory()val_y = torch.from_numpy(_v[1: n_val * CFG["seq_len"] + 1]).view(n_val, -1).pin_memory()log(f"val set: {len(val_tokens)/1e6:.2f}M tokens -> {n_val} sequences")@torch.no_grad()def evaluate(max_batches=None):    """Mean val loss over an identical batch count on every rank.    The count has to match: the result is combined with an AVG all-reduce, so a    rank that ran out of sequences one batch earlier would fold a mean over a    different sample -- and a rank that got zero batches would fold in a 0.0 and    halve the reported loss.    """    model.eval()    bs = CFG["micro_bs"]    n_avail = n_val // (WORLD * bs)    n_group = min(n_avail, max_batches) if max_batches else n_avail    # Spread the sampled groups over the WHOLE set, do not take a prefix.    # build_val_tokens lays the buckets out end to end -- English first, then one    # per German source -- so `val_batches` groups off the front is an    # English-only metric, which is precisely the bug the balanced set exists to    # prevent: it once made best-checkpoint selection prefer a model 0.44 nats    # worse on English. Evenly spaced indices are deterministic, so successive    # evaluations stay comparable.    # spaced from the first group to the last inclusive, so the final bucket --    # whichever source happens to be last -- is not the one that gets dropped    groups = ([0] if n_group == 1 else              [(g * (n_avail - 1)) // (n_group - 1) for g in range(n_group)]) if n_group else []    losses = []    for j in groups:        start = (j * WORLD + RANK) * bs        x = val_x[start:start + bs].to(DEV, non_blocking=True)        y = val_y[start:start + bs].to(DEV, non_blocking=True)        with torch.autocast("cuda", dtype=torch.float16):            losses.append(model(x, y).float())    out = torch.stack(losses).mean() if losses else torch.zeros((), device=DEV)    if WORLD > 1:        dist.all_reduce(out, op=dist.ReduceOp.AVG)    model.train()    return out.item()# --------------------------------------------------------------------------- ## schedules# --------------------------------------------------------------------------- #def lr_mult(step, total):    """Trapezoid: linear warmup -> flat -> linear cooldown over the last    `cooldown_frac` of training."""    warm = CFG["warmup_steps"]    if step < warm:        return (step + 1) / warm    prog = (step - warm) / max(1, total - warm)    cool = CFG["cooldown_frac"]    if prog < 1 - cool:        return 1.0    return max(CFG["final_lr_frac"], (1 - prog) / cool)def in_cooldown(step, total):    """Is `step` inside the LR cooldown of a `total`-step schedule?    Not `lr_mult(...) < 1.0`: that is also true during the warmup, which would make    the plateau latch fire at step 0 of a fresh run and never again.    """    warm = CFG["warmup_steps"]    return (step >= warm            and (step - warm) / max(1, total - warm) >= 1 - CFG["cooldown_frac"])def muon_momentum(step):    warm = max(1, CFG["momentum_warmup_steps"])    return min(CFG["muon_momentum"], 0.85 + (CFG["muon_momentum"] - 0.85) * step / warm)TOKENS_PER_STEP = CFG["micro_bs"] * CFG["seq_len"] * CFG["grad_accum"] * WORLDdef flops_per_token(seq):    """fwd+bwd FLOPs/token: 6*N over the weight matmuls (the embedding lookup is    free, the lm_head is not) plus the causal QK^T / AV matmuls, which depend on    the sequence length actually in use -- during the short-sequence warmup that    is seq_len/2, and reporting MFU against the full length overstates it."""    return (6 * (n_params - raw_model.embed.weight.numel())            + 6 * CFG["n_layer"] * CFG["dim"] * seq)FLOPS_PER_TOKEN = flops_per_token(CFG["seq_len"])T4_PEAK = 65e12   # fp16 tensor-core peak of one T4def bits_per_char(loss):    """Cross-entropy in bits per *character* rather than per token.    A change of tokenizer -- which a change of corpus forces -- makes token-level    losses incomparable between runs: a vocabulary that packs German more    tightly reports a higher loss for the same model. Dividing by the measured    chars/token (written into the config by tokenizer_train.report) removes that,    and is the only honest way to say whether a new data mixture actually helped.    """    return loss / math.log(2) / max(1e-9, CFG["chars_per_token"])# --------------------------------------------------------------------------- ## resume# --------------------------------------------------------------------------- ## Three states worth keeping, each with its own sharded Muon slice. Muon's momentum# buffers are sharded -- rank r only ever holds state for the parameters it# orthogonalises -- so a rank-0-only checkpoint would silently drop half of them on# resume and each rank persists its own.##   ""        the rolling checkpoint, written on a step counter.#   "best"    only moves when validation improves. The rolling one would otherwise be#             overwritten by a divergence within save_every steps and the good state#             would be gone.#   "plateau" written once, at the step the LR cooldown begins. This is the state a#             WSD/trapezoid run can be *extended* from for free: the LR is still at#             the plateau there, so a longer plan simply continues and cools down at#             its own end. Resuming the cooled model instead means re-warming one#             that has already settled -- the loss spikes and the result ends up#             behind a single continuous run of the same total length.CKPT_NAMES = {"": ("ckpt.pt", "muon_rank{r}.pt"),              "best": ("ckpt_best.pt", "muon_best_rank{r}.pt"),              "plateau": ("ckpt_plateau.pt", "muon_plateau_rank{r}.pt")}def ckpt_paths(tag):    main, muon = CKPT_NAMES[tag]    return (os.path.join(CFG["out_dir"], main),            os.path.join(CFG["out_dir"], muon.format(r=RANK)))CKPT, MUON_CKPT = ckpt_paths("")BEST_CKPT, BEST_MUON = ckpt_paths("best")PLATEAU_CKPT, PLATEAU_MUON = ckpt_paths("plateau")_tag_of_file = {main: tag for tag, (main, _) in CKPT_NAMES.items()}if CFG["resume_from"] not in _tag_of_file:    log(f"ERROR: resume_from={CFG['resume_from']!r} is not one of {sorted(_tag_of_file)}")    sys.exit(1)RESUME_TAG = _tag_of_file[CFG["resume_from"]]RESUME_CKPT, RESUME_MUON = ckpt_paths(RESUME_TAG)metrics_path = os.path.join(CFG["out_dir"], "metrics.jsonl")def load_muon(path):    """Load Muon state, then restore any hyperparameter the saved groups predate.    torch's Optimizer.load_state_dict replaces each param_group wholesale with the    saved one, keeping only `params`. A checkpoint written before `weight_decay`    existed therefore produces groups without that key, and the next step() dies    with a KeyError. Re-seed anything missing from the live defaults.    """    opt_muon.load_state_dict(torch.load(path, map_location=DEV, weights_only=False))    for g in opt_muon.param_groups:        for k, v in opt_muon.defaults.items():            g.setdefault(k, v)        g.setdefault("numel", g["params"][0].numel())        if g["numel"] not in opt_muon._bufs:            buf = torch.empty(WORLD, g["numel"], dtype=NS_DTYPE, device="cuda")            opt_muon._bufs[g["numel"]] = (buf, [buf[i] for i in range(WORLD)])start_step, total_steps = 0, CFG["total_steps"] or 10**9calibrated = CFG["total_steps"] is not None# The LR schedule is fitted to the whole *plan*, not to this session, so a run split# over several sessions follows one continuous trapezoid with a single cooldown at# the end instead of cooling down every time and throwing the momentum away.## The plan is a number of TRAINING hours, decided once by the notebook, and it is# the only duration in here that is configured. There is no per-session budget: a# session trains until the Kaggle clock or the plan runs out, whichever comes first# (`session_budget_h` is that clock, resolved by the notebook just before launch).# Fixed session lengths used to need a margin so the last cooldown could finish# inside its own clock, and a grace extension for when it did not; with the session# clock as the only limit, a last cooldown that runs a few minutes over simply ends# in the next session -- which the SFT behind it needs anyway.sessions_done = 0elapsed_h = 0.0PLAN_H = float(CFG["plan_h"])# How far short of the plan still counts as "the plan is done" once the schedule# itself has finished: more than the fit's accounting error (~0.07h measured), and# far less than any deliberate enlargement of `plan_h`, which must still resume.PLAN_SLACK_H = 0.5if CFG["resume"] and os.path.exists(RESUME_CKPT):    blob = torch.load(RESUME_CKPT, map_location=DEV, weights_only=False)    raw_model.load_state_dict(blob["model"])    opt_adam.load_state_dict(blob["opt_adam"])    scaler.load_state_dict(blob["scaler"])    if os.path.exists(RESUME_MUON):        load_muon(RESUME_MUON)    else:        log(f"WARNING: {RESUME_MUON} missing, Muon momentum restarts from zero")    # A checkpoint is only continuable under the tokenizer that produced it. Row i of    # the embedding means token i and nothing else, and because the vocabulary is a    # fixed 32768 either way, a mismatched pair loads *without error* and trains on    # happily while every embedding means the wrong string. The way to walk into this    # is a SMOKE_TEST run in the same notebook: it writes the same ckpt.pt filenames    # but a tokenizer_smoke.json, so the real run afterwards would retrain the    # tokenizer, find that checkpoint and resume from 40 smoke steps.    _ckpt_sha = blob.get("tok_sha")    if _ckpt_sha is None:        log(f"WARNING: {RESUME_CKPT} predates the tokenizer fingerprint; it cannot be "            f"checked against {os.path.basename(CFG['tokenizer_path'])}")    elif _ckpt_sha != _tok_key:        log(f"ERROR: {os.path.basename(RESUME_CKPT)} was trained with tokenizer "            f"{_ckpt_sha}, but {os.path.basename(CFG['tokenizer_path'])} hashes to "            f"{_tok_key}. Every embedding row would mean a different token, and the "            f"shapes match so nothing would complain.\n"            f"  -> after a SMOKE_TEST run in this notebook: delete ckpt*.pt, muon_*.pt "            f"and model_final.pt (the smoke run wrote them), then Run All again\n"            f"  -> after changing the corpus or vocab_size: the same, this is a new run\n"            f"  -> to continue the old model: restore its tokenizer.json")        sys.exit(1)    start_step, calibrated = blob["step"] + 1, True    sessions_done, elapsed_h = blob.get("sessions_done", 0), blob.get("elapsed_h", 0.0)    # An explicit CONFIG["total_steps"] must win, otherwise extending a finished    # run is impossible: the checkpoint's own total is already reached, the loop    # never executes, and the script silently does nothing but a final eval.    total_steps = CFG["total_steps"] or blob["total_steps"]    lr_scale = CFG["resume_lr_scale"]    log(f"resumed from {RESUME_CKPT} at step {start_step} of {total_steps}")    if RESUME_TAG == "plateau" and CFG["total_steps"] is None \            and elapsed_h < PLAN_H - PLAN_SLACK_H:        # Branching a bigger plan off the plateau. Its `total_steps` belongs to the        # old, shorter schedule, and following that would start cooling down        # immediately; drop it and re-fit to the whole remaining plan instead. The LR        # needs no scaling at all here -- that is the entire point of the plateau        # snapshot, and why `resume_lr_scale` is only for the cooled checkpoint.        log(f"branching from the plateau: dropping the old {total_steps}-step schedule "            f"and re-fitting to the {PLAN_H - elapsed_h:.2f}h the plan has left")        calibrated, total_steps = False, 10**9    if start_step >= total_steps and CFG["total_steps"] is None \            and elapsed_h < PLAN_H - PLAN_SLACK_H:        # The checkpoint finished the plan it was trained under, but the plan has        # since been enlarged (`plan_h`, usually via PLAN). Re-measure and re-fit        # rather than refusing: "raise the plan and press Run All" should simply work.        log(f"this checkpoint finished a {total_steps}-step schedule after {elapsed_h:.2f}h, "            f"but the plan now allows {PLAN_H:.2f}h -- recalibrating to continue.")        if lr_scale == 1.0:            # Its cooldown has already run, so the weights sit where a decayed LR            # left them; coming straight back to the peak spikes the loss for            # hundreds of steps. Applied automatically rather than left as a knob:            # this branch detects the exact situation the knob exists for, and the            # user has already had to remember one setting to get here.            lr_scale = 0.4            log(f"its LR cooldown had already run, so the continuation peak is scaled to "                f"{lr_scale} to avoid the loss spike (set resume_lr_scale to override)")        calibrated, total_steps = False, 10**9    elif start_step >= total_steps:        log(f"ERROR: this checkpoint has already finished its schedule "            f"({start_step - 1} of {blob['total_steps']} steps) and the plan has no hours "            f"left ({elapsed_h:.2f}h of {PLAN_H:.2f}h). Raise plan_h to train "            f"further, or set CONFIG['total_steps'] explicitly -- and see resume_lr_scale, "            f"because the cooldown has already run.")        sys.exit(1)    if lr_scale != 1.0:        # Continuing past a completed cooldown means re-warming the LR. Coming        # straight back to the original peak spikes the loss; a reduced peak picks        # up closer to where the model actually is.        base_lrs = [[lr * lr_scale for lr in group] for group in base_lrs]        log(f"continuation LRs scaled by {lr_scale} "            f"(muon {CFG['lr_muon'] * lr_scale:.4f})")def save_ckpt(step, total, tag=""):    """Weights + Adam + scaler on the master, Muon's sharded slice on every rank.    The barrier makes the *order* of the writes deterministic, which matters    because a session can be killed in the middle of one. Without it rank 1 might    still be writing its Muon slice when rank 0 has already renamed ckpt.pt into    place: the resume then pairs weights from step N with momentum from step    N - save_every, and Muon turns a stale buffer into a full-magnitude update in    a direction the model has long left. With it, ckpt.pt appearing means every    Muon slice is already at least that new -- the safe direction, since momentum    that is slightly ahead of the weights is just momentum.    """    main_path, muon_path = ckpt_paths(tag)    torch.save(opt_muon.state_dict(), muon_path + ".tmp")    os.replace(muon_path + ".tmp", muon_path)    if WORLD > 1:        dist.barrier()    if not MASTER:        return    torch.save(dict(model=raw_model.state_dict(), opt_adam=opt_adam.state_dict(),                    scaler=scaler.state_dict(), step=step, total_steps=total,                    sessions_done=sessions_done + 1, elapsed_h=plan_elapsed_h(),                    tok_sha=_tok_key, cfg=CFG), main_path + ".tmp")    os.replace(main_path + ".tmp", main_path)def restore_ckpt(main_path, muon_path):    """Roll back to a saved state and reset everything that could still be NaN."""    blob = torch.load(main_path, map_location=DEV, weights_only=False)    raw_model.load_state_dict(blob["model"])    opt_adam.load_state_dict(blob["opt_adam"])    if os.path.exists(muon_path):        load_muon(muon_path)    # A single NaN gradient poisons a Muon momentum buffer for good, and the    # buffers are what the checkpoint carries forward. Zero them rather than    # trusting them.    for st in opt_muon.state.values():        if "momentum_buffer" in st:            st["momentum_buffer"].zero_()    scaler.load_state_dict(dict(scale=CFG["recovery_loss_scale"], growth_factor=2.0,                                backoff_factor=0.5, growth_interval=500, _growth_tracker=0))    return blob["step"]# --------------------------------------------------------------------------- ## training loop# --------------------------------------------------------------------------- #log(f"tokens/step {TOKENS_PER_STEP:,} = {CFG['micro_bs']} x {CFG['seq_len']} x "    f"{CFG['grad_accum']} accum x {WORLD} gpus")log(f"plan: {PLAN_H:.2f}h of training, {elapsed_h:.2f}h done over {sessions_done} "    f"session(s), this session has {CFG['session_budget_h']:.2f}h | model FLOPs/token "    f"{FLOPS_PER_TOKEN/1e6:.0f}M | "    + ("auto-calibrating the step count" if not calibrated else       f"fixed {total_steps} steps" if CFG["total_steps"] else       f"resuming a {total_steps}-step schedule, refitted to the plan on the LR plateau"))deadline = time.time() + CFG["session_budget_h"] * 3600step_times: list[float] = []tokens_done = start_step * TOKENS_PER_STEPskipped = 0t_start = time.time()model.train()step = start_stepfull_seq_since = None      # first step at the full sequence length (latched)consec_skips = 0           # optimizer steps skipped in a row (NaN/inf gradients)recoveries = 0best_val = float("inf")annealed = False# Latched from where this session starts, so a session that resumes inside the# cooldown leaves the existing snapshot alone instead of overwriting it with a# state that is no longer on the plateau.plateau_saved = calibrated and in_cooldown(start_step, total_steps)gave_up = False             # the fp16 watchdog ran out of recoveries# Anchor for the realised rate: wall-clock seconds and steps since the last point# where the cost of a step changed. Re-anchored when the sequence length doubles,# because T=512 steps are cheaper and averaging the two shapes together would make# the fit optimistic for the rest of the run.rate_t0, rate_step0 = None, NoneMIN_RATE_WINDOW = 100        # below this a single stall would dominate the averagedef realised_sec_per_step(fallback):    """Wall-clock seconds per step including everything the deadline pays for."""    if rate_t0 is None or step - rate_step0 < MIN_RATE_WINDOW:        return fallback, False    return (time.time() - rate_t0) / (step - rate_step0), Truedef plan_elapsed_h():    return elapsed_h + (time.time() - t_start) / 3600def fit_total_steps(from_step, sec_per_step):    """Steps that still fit in the plan's remaining *training* hours.    `sec_per_step` must be the **realised** wall-clock cost of a step, not the    median of `step_times`. `dt` is measured around the optimizer step only, while    the deadline this is fitted against also pays for validation, checkpoint    writes, the annealing loader swap and every data stall -- about 2% of a    session at the default val_every/save_every. Fitting against the step time    alone therefore promises ~2% more steps than the hours hold, and the schedule    is cut off short of its cooldown. `realised_sec_per_step()` supplies the honest    number once there is enough history to measure one.    Kaggle caps a session at 12h and a 125M model needs more than one to reach    Chinchilla's 20 tokens/parameter, so the LR trapezoid spans the whole plan and    each session runs its slice of one continuous schedule -- fitting it per    session cools the LR to zero at the end of every one, which is what    `resume_lr_scale` used to work around.    Counted against `elapsed_h`, which measures time actually spent in this loop,    rather than against a session counter. A session counter charges a full    session for one that was interrupted after twenty minutes: two early    interrupts of a two-session plan left the third session believing it was the    last, and it fitted the whole remaining schedule into itself -- 2.24B tokens    where the plan asked for 3.09B. Measuring the hours actually trained is    immune to that, and it also keeps `total_steps` steady from session to    session instead of jumping every time a new one starts.    """    left = max(0.0, (PLAN_H - plan_elapsed_h()) * 3600)    fitted = from_step + max(50, int(left / sec_per_step))    # Floor: never shorten the schedule past the point where the cooldown would    # have to begin at the current step. Without it, a checkpoint carried into a    # smaller plan makes `left` zero, the fit collapses to +50 steps, and the LR    # falls from 1.0 to final_lr_frac between two steps -- a run that reports    # success having done no cooldown at all. Recalibration only happens while    # mult >= 1.0, i.e. at or before the cooldown start, so this floor can never    # push the schedule past where it already was.    floor = math.ceil((from_step - CFG["warmup_steps"])                      / max(1e-9, 1 - CFG["cooldown_frac"])) + CFG["warmup_steps"]    # ...but capped by the schedule already in force, so the floor can only stop a    # collapse, never extend the plan. Uncapped it runs away: once the run is past    # PLAN_H, `left` is zero and the floor alone would raise total_steps a little    # further at every recalibration, and the plan would never end.    return max(fitted, min(floor, total_steps), from_step + 50)def anneal_step(total):    """First step of the high-quality cooldown mixture, or None if disabled."""    return int(total * (1 - CFG["anneal_frac"])) if CFG["anneal_frac"] else Nonedef switch_loader(overrides, why):    """Rebuild the input pipeline mid-run.    Used once, to swap in the annealing mixture. The old workers have to be gone    before the new ones start: each holds a ~1 GB arrow table, and two    generations alive at once is how this host gets OOM-killed. Costs one refill    stall (a German row group is ~40 s to first byte), which is why    nccl_timeout_min is 60 and not the NCCL default of 10.    """    global loader, train_iter    log(f"[data] {why}")    train_iter = None    if loader is not None:        del loader    gc.collect()    loader = make_loader(overrides)    train_iter = iter(loader)# Where in the corpora this session reads. Every stream is infinite by# reshuffling and replaying its files, but from a fixed seed -- so without this# each session replayed the identical document order, and since one session gets# through well under one parquet file per shard, a two-session plan trained twice# on the first half of the data instead of once on all of it. Keyed on the resume# step so a crash-restart from the same checkpoint is still reproducible.data_cfg["stream_seed"] = start_step# The annealing mixture is a *subset* of the same sources -- German is the same# file list either way -- so reusing the offset would make the cooldown replay the# documents this session already trained on, which is the one stretch of the run# where that matters most.ANNEAL["stream_seed"] = start_step + 104729if start_step:    log(f"data stream offset {start_step} (this session reads different documents "        f"than the last one)")# The data stream is built here rather than at import time: the schedule is only# known after the resume block, and starting a worker before the validation set# is tokenised would put a 1 GB arrow table next to that peak for no reason._resume_into_anneal = anneal_step(total_steps) is not None and \    start_step >= anneal_step(total_steps)if _resume_into_anneal:    annealed = Trueswitch_loader(ANNEAL if _resume_into_anneal else None,              "high-quality annealing mixture (resumed inside the cooldown)"              if _resume_into_anneal else "training mixture")while step < total_steps:    t_step = time.time()    frac = step / max(1, total_steps)    # Short-sequence warmup: (B, 2T) packed rows reshaped to (2B, T) are still    # valid packed sequences, so tokens/step is identical and attention halves.    # Latched: total_steps can shrink on recalibration, and flipping back to    # short sequences would just cost another compile.    if full_seq_since is None and frac >= CFG["short_seq_frac"]:        full_seq_since = step        rate_t0, rate_step0 = time.time(), step      # T doubles; the old rate is stale    seq_now = CFG["seq_len"] if full_seq_since is not None else CFG["seq_len"] // 2    mult = lr_mult(step, total_steps)    if CFG["save_plateau"] and calibrated and not plateau_saved \            and in_cooldown(step, total_steps):        # The last state on the plateau, kept so the plan can be enlarged later        # without a re-warm. Written before the step runs, so it is exactly the        # weights the cooldown starts from.        plateau_saved = True        save_ckpt(step - 1, total_steps, tag="plateau")        log(f"[plateau] saved ckpt_plateau.pt at step {step} ({plan_elapsed_h():.2f}h of "            f"training, {tokens_done/1e9:.2f}B tokens) -- the LR cooldown starts here, so "            f"this is the state to resume from if you ever extend the plan "            f"(resume_from='ckpt_plateau.pt' + a larger plan_h)")    for opt, bases in zip(optimizers, base_lrs):        for g, base in zip(opt.param_groups, bases):            g["lr"] = base * mult    for g in opt_muon.param_groups:        g["momentum"] = muon_momentum(step)    for micro in range(CFG["grad_accum"]):        batch = next_batch().to(DEV, non_blocking=True)        inp, tgt = batch[:, :-1], batch[:, 1:]        if seq_now != CFG["seq_len"]:            inp, tgt = inp.reshape(-1, seq_now), tgt.reshape(-1, seq_now)        sync_ctx = (model.no_sync() if (WORLD > 1 and micro < CFG["grad_accum"] - 1)                    else contextlib.nullcontext())        with sync_ctx:            with torch.autocast("cuda", dtype=torch.float16):                loss = model(inp, tgt)            scaler.scale(loss / CFG["grad_accum"]).backward()    for opt in optimizers:        scaler.unscale_(opt)    gnorm = torch.nn.utils.clip_grad_norm_(all_params, CFG["grad_clip"])    ok = torch.isfinite(gnorm).to(torch.float32)    if WORLD > 1:        dist.all_reduce(ok, op=dist.ReduceOp.MIN)   # every rank must make the same call    if ok.item() == 1.0:        for opt in optimizers:            opt.step()        consec_skips = 0    else:        skipped += 1        consec_skips += 1    scaler.update()    pinned = clamp_loss_scale()    for opt in optimizers:        opt.zero_grad(set_to_none=True)    # Watchdog. Without it a collapsed loss scale is silent: gradients are NaN,    # every step is skipped, the weights never move, and the run happily burns    # the rest of an 8h budget reporting a plausible-looking loss.    if consec_skips >= CFG["max_consecutive_skips"]:        src = BEST_CKPT if os.path.exists(BEST_CKPT) else CKPT        muon_src = BEST_MUON if os.path.exists(BEST_CKPT) else MUON_CKPT        recoveries += 1        log(f"!! {consec_skips} consecutive skipped steps at step {step} "            f"(scale {scaler.get_scale():g}{', pinned at floor' if pinned else ''}) "            f"-- training is not progressing.")        if recoveries > CFG["max_recoveries"] or not os.path.exists(src):            log(f"!! giving up after {recoveries - 1} recoveries. "                f"Lower lr_muon, or lower relu2_clamp, and restart.")            gave_up = True            break        restored = restore_ckpt(src, muon_src)        consec_skips = 0        log(f"!! rolled back to step {restored} from {os.path.basename(src)}, "            f"Muon momentum zeroed, loss scale reset to {scaler.get_scale():g} "            f"(recovery {recoveries}/{CFG['max_recoveries']})")    torch.cuda.synchronize()    dt = time.time() - t_step    step_times.append(dt)    tokens_done += TOKENS_PER_STEP    if step - start_step == 3:   # after compile, before the timing window        torch.cuda.reset_peak_memory_stats()    if step - start_step == CFG["calibration_steps"]:        # Start measuring the realised rate here, past the compile spike. Set on        # every session, resumed ones included -- they skip the calibration below        # but still need the anchor for their recalibrations.        rate_t0, rate_step0 = time.time(), step        log(f"peak GPU memory {torch.cuda.max_memory_allocated()/2**30:.2f} GiB allocated / "            f"{torch.cuda.max_memory_reserved()/2**30:.2f} GiB reserved of "            f"{torch.cuda.get_device_properties(LOCAL_RANK).total_memory/2**30:.1f} GiB")    if not calibrated and step - start_step == CFG["calibration_steps"]:        med = float(np.median(step_times[-max(5, CFG["calibration_steps"] // 2):]))        total_steps = int(agree(fit_total_steps(step, med))[0])        calibrated = True        log(f"calibrated: {med*1000:.0f} ms/step at T={seq_now} -> total_steps={total_steps} "            f"(~{total_steps*TOKENS_PER_STEP/1e9:.2f}B tokens)")    # The first calibration measured the cheap short-sequence phase, and step time    # drifts up ~15% as the T4s heat up. Re-fit on the LR plateau (never during    # warmup or cooldown, where changing total_steps would move the LR) so the    # cooldown is not truncated by the wall-clock deadline.    due = (full_seq_since is not None and step - full_seq_since == 20) or \          (step > start_step and (step - start_step) % CFG["recalibrate_every"] == 0)    if CFG["total_steps"] is None and calibrated and due and mult >= 1.0:        med = float(np.median(step_times[-min(len(step_times), 50):]))        sec, measured = realised_sec_per_step(med)        new_total = int(agree(fit_total_steps(step, sec))[0])        if abs(new_total - total_steps) > 0.02 * total_steps:            log(f"recalibrated at T={seq_now}: {med*1000:.0f} ms/step, "                f"{sec*1000:.0f} ms/step realised"                f"{'' if measured else ' (estimated, window too short)'} -> "                f"total_steps={new_total} (~{new_total*TOKENS_PER_STEP/1e9:.2f}B tokens)")        total_steps = new_total    # The last `anneal_frac` of the plan trains on a higher-quality mixture: the    # FineWeb-Edu classifier threshold goes up and German Commons is narrowed to    # its curated sources. Standard practice (Llama-3, MiniCPM, SmolLM all do a    # version of it) and it lands inside the LR cooldown, where the model is    # consolidating rather than exploring. The set is kept deliberately broad --    # 14 German sources, not two -- because narrowing the late-training    # distribution is exactly what caused this project's forgetting bug once    # already; `diagnose2.py` is the tool to check for it.    if not annealed and calibrated and anneal_step(total_steps) is not None \            and step >= anneal_step(total_steps):        annealed = True        switch_loader(ANNEAL, f"annealing from step {step}: FineWeb-Edu score >= "                              f"{CFG['anneal_min_edu_score']}, mixture "                              + ", ".join(f"{n} {w:g}" for n, w                                          in ANNEAL["source_weights"].items()))    if step % CFG["log_every"] == 0 or step == total_steps - 1:        tps = TOKENS_PER_STEP / dt        # ETA to the end of the PLAN, which spans several sessions -- so it can and        # should exceed the 12h session cap. Labelled to stop it reading as "this        # session will overrun".        eta = min(99.0, (total_steps - step) * float(np.median(step_times[-50:])) / 3600)        log(f"step {step:6d}/{total_steps if calibrated else '?':>6} loss {loss.item():6.4f} "            f"lr {mult:5.3f} gnorm {float(gnorm):6.3f} scale {scaler.get_scale():>7.0f} "            f"{dt*1000:6.0f}ms {tps:7.0f} tok/s "            f"mfu {flops_per_token(seq_now)*tps/(T4_PEAK*WORLD):5.1%} "            f"T={seq_now} plan-eta {eta:4.2f}h")        if MASTER:            with open(metrics_path, "a") as fh:                fh.write(json.dumps(dict(step=step, loss=loss.item(), lr_mult=mult, tok_s=tps,                                         dt=dt, tokens=tokens_done, seq=seq_now)) + "\n")    if CFG["val_every"] and step > start_step and step % CFG["val_every"] == 0:        with torch.no_grad():            now = torch.stack([p.detach().float().norm() for p in hidden_p])            growth = now[_grow_mask] / init_norms        log(f"    hidden-weight norms: growth vs init mean {growth.mean():.1f}x "            f"max {growth.max():.1f}x (over {int(_grow_mask.sum())} of {len(hidden_p)} "            f"matrices; the rest are zero-init) | zero-init mean norm "            f"{now[~_grow_mask].mean():.2f}")        vl = evaluate(CFG["val_batches"])        mark = ""        if math.isfinite(vl) and vl < best_val:            best_val, mark = vl, "  <- best"            save_ckpt(step, total_steps, tag="best")        log(f"    val loss {vl:.4f}   ppl {math.exp(min(vl, 20)):.2f}   "            f"{bits_per_char(vl):.3f} bits/char{mark}")        if MASTER:            with open(metrics_path, "a") as fh:                fh.write(json.dumps(dict(step=step, val_loss=vl, tokens=tokens_done,                                         bpc=bits_per_char(vl), best_val=best_val)) + "\n")    # Never checkpoint over a good state while the gradients are non-finite --    # that is how a divergence erases the last usable model.    if (CFG["save_every"] and step > start_step and step % CFG["save_every"] == 0            and consec_skips == 0):        save_ckpt(step, total_steps)    step += 1    # Rank 0's verdict on every rank, every step -- see agree().    stop = time.time() > deadline - CFG["reserve_seconds"]    if agree(stop)[0]:        log(f"session clock reached at step {step} of {total_steps}")        break# --------------------------------------------------------------------------- ## finish# --------------------------------------------------------------------------- ## Shut the streaming workers down before the final eval: they hold open HTTP# connections and a few GB of parquet row-group buffers, and nothing below needs# them.train_iter = Nonedel loadergc.collect()vl = evaluate(CFG["val_batches"] * 4 if CFG["val_batches"] else None)if math.isfinite(best_val) and vl > best_val + 0.05 and os.path.exists(BEST_CKPT):    log(f"final state ({vl:.4f}) is worse than the best checkpoint ({best_val:.4f}); "        f"exporting the best one instead")    restore_ckpt(BEST_CKPT, BEST_MUON)    vl = evaluate(CFG["val_batches"] * 4 if CFG["val_batches"] else None)finished = step >= total_stepslog(f"done: {step - start_step} steps, {tokens_done/1e9:.3f}B tokens in "    f"{(time.time()-t_start)/3600:.2f}h ({plan_elapsed_h():.2f}h of training over "    f"{sessions_done + 1} session(s)) | {skipped} fp16-overflow steps skipped | "    f"final val loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.2f})")if gave_up:    log(f"\n*** TRAINING STOPPED: the fp16 watchdog ran out of recoveries at step {step}. "        f"Running this again unchanged will hit the same wall -- see the lines above. ***\n")elif not finished:    # The normal end of every session but the last. This used to be printed as    # "PLAN NOT FINISHED ... this checkpoint is worse", which on a clean end of    # session 1 of 3 reads exactly like a crash -- and was taken for one.    left_h = max(0.0, PLAN_H - plan_elapsed_h())    log(f"\n=== session done as planned: step {step} of {total_steps}, "        f"{plan_elapsed_h():.2f}h of the {PLAN_H:.2f}h plan trained "        f"({plan_elapsed_h() / max(1e-9, PLAN_H):.0%}), ~{left_h:.1f}h to go.\n"        f"    The LR is still on its plateau, which is correct mid-plan: the cooldown "        f"runs at the end of the last session.\n"        f"    Next: a new Kaggle session, Run All, unchanged. ===\n")save_ckpt(step - 1, total_steps)if MASTER:    torch.save(dict(model=raw_model.state_dict(), cfg=CFG, pad_vocab=PAD_VOCAB,                    real_vocab=REAL_VOCAB, val_loss=vl, steps=step, tokens=tokens_done),               os.path.join(CFG["out_dir"], "model_final.pt"))    with open(metrics_path, "a") as fh:        fh.write(json.dumps(dict(step=step, val_loss=vl, tokens=tokens_done, final=True)) + "\n")if WORLD > 1:    dist.barrier()    dist.destroy_process_group()

In [ ]:
%%writefile /kaggle/working/chatdata.py"""Bilingual (EN/DE) instruction and preference data for the post-training stages.Unlike pretraining, this is *small* data -- the whole SFT mixture is under agigabyte -- so it is loaded with `datasets.load_dataset` and tokenised once intoa flat array on disk. The pretraining pipeline avoids `datasets` because itsstreaming mode shards itself per DataLoader worker (see dataio.py); none of thatapplies here, and a materialised corpus buys exact epoch boundaries, a shufflethat actually shuffles, and a training loop with no tokeniser in it.Sources-------English SFT is `HuggingFaceTB/smol-smoltalk` in one piece: the SmolTalk variantHugging Face cut down for models under 1B and used to build SmolLM2-135M-Instruct,which is exactly this size. It is 59% shortened Smol-Magpie-Ultra -- the subsetSmolTalk's own card calls "the core component of our mix" and reports as beatingOpenHermes -- plus constraints, rewriting, summarisation and 48k code-instructionrows. An earlier hand-picked selection of smoltalk subsets left Magpie-Ultra outand kept `openhermes-100k` instead, which is the opposite of that finding.German SFT is the German fifth of `smoltalk2`'s multilingual no-think subset:prompts translated from Magpie-Ultra and Smol-Constraints, but the **answersgenerated in German**, which is what keeps translationese out of what the modellearns to write. 52,743 conversations, ~25M tokens. The rest is`alpaca-gpt4_de`, `oasst_de` and DiscoResearch's grounded-QA set.The `mayflowergmbh` translations that used to make up 86% of the German SFTtokens are gone, measured defect by measured defect: 25% of `ultra-chat_de`'suser turns are follow-ups whose conversation history was stripped when it wasflattened to single turns ("Vielen Dank fuer die Statistiken ..."), and a modeltrained on them answers a bare "Kannst du mir mehr Details dazu geben?" byinventing a context it was never given -- which the first run did, verbatim.`dolphin_de` translated FLAN tasks into incoherence (a translation exercisebecame "Schritt 1: Verstehen Sie den gegebenen galicischen Satz" with noGalician sentence in sight), `evol-instruct_de` carries untranslated Englishmid-sentence, and `OpenSchnabeltier` is 95% competition maths, the very thingthe English side drops on purpose.Preference data is UltraFeedback for English -- the set Hugging Face ran DPO onfor SmolLM2-135M-Instruct -- and the three German DPO sets that exist at usablesize. `train_prefs` is the split with the binarised pairs."""from __future__ import annotationsimport randomimport reimport numpy as np_WORD = re.compile(r"[a-zA-ZÀ-ÿ]+")IM_START = "<|im_start|>"IM_END = "<|im_end|>"# --------------------------------------------------------------------------- ## chat template# --------------------------------------------------------------------------- #def ensure_chat_tokens(tok, log=print) -> int:    """Make sure `<|im_start|>` / `<|im_end|>` exist; return the new vocab size.    Tokenizers trained after these were reserved in `tokenizer_train.CHAT_TOKENS`    already contain them and this is a no-op. For an older one the tokens are    appended, which pushes the vocabulary past its 128-aligned padding -- the    caller then has to grow the embedding and the head, which `grow_vocab` does.    """    missing = [t for t in (IM_START, IM_END) if tok.token_to_id(t) is None]    if missing:        tok.add_special_tokens(missing)        log(f"added chat tokens {missing} -> vocab {tok.get_vocab_size()}")    return tok.get_vocab_size()def grow_vocab(model, new_size: int):    """Extend `embed` and `lm_head` to `new_size` rows, keeping the old ones.    New embedding rows get the same small uniform init as the original ones; new    head rows are zeroed, which is where the head started anyway, so the added    tokens begin at the same logit as an untrained one rather than at whatever    uninitialised memory held.    """    import torch    old, dim = model.embed.weight.shape    if new_size <= old:        return model    dev = model.embed.weight.device    emb = torch.empty(new_size, dim, device=dev).uniform_(-0.5 * dim**-0.5, 0.5 * dim**-0.5)    emb[:old] = model.embed.weight.data    model.embed.weight.data = emb    model.embed.num_embeddings = new_size    head = torch.zeros(new_size, dim, device=dev)    head[:old] = model.lm_head.weight.data    model.lm_head.weight.data = head    model.lm_head.out_features = new_size    return modelclass ChatFormat:    """ChatML, tokenised segment by segment.    Building the ids from segments rather than encoding one rendered string is    what makes the loss mask exact: the model is trained on assistant content and    its terminating `<|im_end|>`, and on nothing else -- not on the role headers,    not on the user's text. Training on the header would teach it to predict    "user" after "<|im_start|>", which is the one thing it must never generate.    """    def __init__(self, tok):        self.tok = tok        self.im_start = tok.token_to_id(IM_START)        self.im_end = tok.token_to_id(IM_END)        assert self.im_start is not None and self.im_end is not None, \            "chat tokens missing -- call ensure_chat_tokens() first"        self.nl = tok.encode("\n").ids        self.head = {r: [self.im_start] + tok.encode(f"{r}\n").ids                     for r in ("system", "user", "assistant")}    def encode(self, messages) -> tuple[list[int], list[int]]:        """-> (ids, loss) with `loss[i] == 1` where the model must predict ids[i]."""        ids: list[int] = []        loss: list[int] = []        for m in messages:            head = self.head.get(m["role"])            if head is None or not m["content"]:                continue            body = self.tok.encode(m["content"]).ids + [self.im_end] + self.nl            ids += head + body            train = 1 if m["role"] == "assistant" else 0            loss += [0] * len(head) + [train] * len(body)        return ids, loss    def encode_batch(self, conversations) -> list[tuple[list[int], list[int]]]:        """`encode` for many conversations at once, row for row the same output.        The corpus build calls the tokenizer once per message, ~1.5M times, on one        core while the other rank waits. The batch path hands every message of a        chunk to the Rust side in one call, which spreads it over all cores.        """        segs = [(i, m) for i, msgs in enumerate(conversations) for m in msgs                if m["role"] in self.head and m["content"]]        bodies = self.tok.encode_batch([m["content"] for _, m in segs])        out = [([], []) for _ in conversations]        for (i, m), enc in zip(segs, bodies):            head = self.head[m["role"]]            body = enc.ids + [self.im_end] + self.nl            ids, loss = out[i]            ids += head + body            train = 1 if m["role"] == "assistant" else 0            loss += [0] * len(head) + [train] * len(body)        return out    def prompt_ids(self, messages) -> list[int]:        """Ids for a conversation prefix, ending where the assistant starts writing.        `messages` is the prompt side only; any assistant turns in it are treated        as conversation history and kept.        """        ids, _ = self.encode(messages)        return ids + self.head["assistant"]# --------------------------------------------------------------------------- ## source registry# --------------------------------------------------------------------------- #def _alpaca(row, sys_field=None, user_field="instruction", extra="input",            out_field="output"):    """instruction/input/output rows. Which field is the system prompt and which    is the user turn is *not* guessable -- dolphin_de and ultra-chat_de put the    system prompt in `instruction`, alpaca-gpt4_de puts the user turn there -- so    every source states its own mapping."""    msgs = []    if sys_field and row.get(sys_field):        msgs.append(dict(role="system", content=row[sys_field]))    user = (row.get(user_field) or "").strip()    if extra and row.get(extra):        user = (user + "\n\n" + row[extra]).strip()    if not user or not row.get(out_field):        return None    msgs.append(dict(role="user", content=user))    msgs.append(dict(role="assistant", content=row[out_field]))    return msgsdef _messages(row, field="messages"):    out = []    for m in row.get(field) or []:        role = m.get("role")        if role in ("system", "user", "assistant") and m.get("content"):            out.append(dict(role=role, content=m["content"]))    return out if len(out) >= 2 else Nonedef _sharegpt(row):    role_of = {"human": "user", "user": "user", "gpt": "assistant",               "assistant": "assistant", "system": "system"}    out = []    for m in row.get("conversations") or []:        role = role_of.get(m.get("from"))        if role and m.get("value"):            out.append(dict(role=role, content=m["value"]))    return out if len(out) >= 2 else Nonedef _oasst_de(row):    """`history` is a list of [user, assistant] pairs preceding the final turn."""    msgs = []    for pair in row.get("history") or []:        if len(pair) == 2 and all(pair):            msgs += [dict(role="user", content=pair[0]),                     dict(role="assistant", content=pair[1])]    tail = _alpaca(row)    return (msgs + tail) if tail else Nonedef _germanrag(row):    """Grounded QA: the passages go in as context, the answer is the target.    `positive_ctx_idx == -1` marks the rows whose contexts do *not* contain the    answer and whose gold answer says so -- those are the valuable ones, because    they are the only German data here that teaches refusing to invent.    """    ctx = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(row.get("contexts") or []))    if not ctx or not row.get("question") or not row.get("answer"):        return None    return [dict(role="user",                 content=f"Beantworte die Frage ausschliesslich anhand der folgenden "                         f"Abschnitte.\n\n{ctx}\n\nFrage: {row['question']}"),            dict(role="assistant", content=row["answer"])]# The multilingual subset carries no language column, so the language has to be# read off the text. Five languages are actually present (de, es, fr, it, pt), and# German is far enough from the other four that counting function words separates# them cleanly: it finds 52,743 German conversations of 254,047, and a hand check# of the samples found no misfiled row._DE_STOP = set("der die das und ist nicht sie ich zu mit den ein eine auf fuer sich es "               "auch von dem des im aber wenn wie".split()) | {"für"}_OTHER_STOP = set("le la les et une des pour que dans qui pas sur vous el los las por con "                  "del se il che di non sono della com uma para nao em the and is of to "                  "you that for with this are".split())def _smoltalk2_de(row):    """German rows of smoltalk2's multilingual subset; everything else -> None."""    msgs = _messages(row)    if not msgs:        return None    words = _WORD.findall(" ".join(m["content"] for m in msgs).lower())    de = sum(w in _DE_STOP for w in words)    return msgs if de > sum(w in _OTHER_STOP for w in words) and de >= 3 else Nonedef _schnabeltier(row):    if not row.get("instruction_de") or not row.get("output_de"):        return None    return [dict(role="user", content=row["instruction_de"]),            dict(role="assistant", content=row["output_de"])]# (repo, config, split, lang, converter, cap). `cap` bounds rows taken from the# source before shuffling, which is how the mixture weights are set.SFT_SOURCES = [    # -- English: the mix Hugging Face used for SmolLM2-135M-Instruct ---------    ("HuggingFaceTB/smol-smoltalk", None, "train", "en", _messages, None),    # -- German: answers generated in German, not translated ------------------    ("hf://datasets/HuggingFaceTB/smoltalk2/SFT/"     "smoltalk_multilingual_8languages_lang_5_no_think-*.parquet",     None, "train", "de", _smoltalk2_de, None),    ("mayflowergmbh/alpaca-gpt4_de", None, "train", "de", _alpaca, None),    ("mayflowergmbh/oasst_de", None, "train", "de", _oasst_de, None),    ("DiscoResearch/germanrag", None, "train", "de", _germanrag, None),]def _pref_ultrafeedback(row):    """`chosen`/`rejected` are whole conversations; the last turn is the response."""    def last(field):        msgs = row.get(field) or []        return msgs[-1]["content"] if msgs and msgs[-1].get("role") == "assistant" else None    c, r = last("chosen"), last("rejected")    if not row.get("prompt") or not c or not r or c == r:        return None    return [dict(role="user", content=row["prompt"])], c, rdef _pref_fields(row, sys_field=None, user_field="input", chosen="chosen", rejected="rejected"):    c, r = row.get(chosen), row.get(rejected)    user = row.get(user_field)    if not user or not c or not r or c == r:        return None    msgs = []    if sys_field and row.get(sys_field):        msgs.append(dict(role="system", content=row[sys_field]))    msgs.append(dict(role="user", content=user))    return msgs, c, rdef _pref_orca_de(row):    """LLaMA-Factory's alpaca-DPO layout: `output` is exactly [chosen, rejected]."""    out = row.get("output") or []    if len(out) != 2 or not row.get("input") or out[0] == out[1]:        return None    msgs = []    if row.get("instruction"):        msgs.append(dict(role="system", content=row["instruction"]))    msgs.append(dict(role="user", content=row["input"]))    return msgs, out[0], out[1]PREF_SOURCES = [    ("HuggingFaceH4/ultrafeedback_binarized", None, "train_prefs", "en",     _pref_ultrafeedback, None),    ("mayflowergmbh/intel_orca_dpo_pairs_de", None, "train", "de", _pref_orca_de, None),    ("aari1995/ultradistil-intel-orca-dpo-de", None, "train", "de",     lambda r: _pref_fields(r, sys_field="system"), None),    ("VAGOsolutions/SauerkrautLM-Fermented-GER-DPO", None, "train", "de",     lambda r: _pref_fields(r, user_field="instruction"), None),]# --------------------------------------------------------------------------- ## loading# --------------------------------------------------------------------------- #def sft_sources_key(cfg) -> str:    """A fingerprint of *which* SFT data a checkpoint was trained on.    `sft_final.pt` existing is not evidence that it was trained on the mixture now    configured, and "auto" would otherwise skip the stage and hand a stale model to    DPO. The first run walked into exactly this shape of problem at the pretraining    stage, which is why `ckpt.pt` records its corpus too.    """    import hashlib    spec = [(repo, config, split, lang) for repo, config, split, lang, _, _ in SFT_SOURCES]    spec.append(("de_ratio", round(float(cfg["de_ratio"]), 4)))    return hashlib.sha1(repr(sorted(map(str, spec))).encode()).hexdigest()[:12]def _load_rows(repo, config, split, convert, cap, cache_dir, log):    """One source -> a list of converted examples. A dead repo is logged and    skipped: losing one of fifteen sources is a worse corpus, losing the run to    a 404 in hour six is a worse day."""    from datasets import load_dataset    try:        if repo.endswith(".parquet"):            # One subset of a repo whose *config* is far too big to resolve:            # load_dataset("HuggingFaceTB/smoltalk2", "SFT", split=...) pulls the            # whole 81 GiB config. Addressing the files directly fetches the 302 MiB            # that are actually wanted.            ds = load_dataset("parquet", data_files=repo, split="train",                              cache_dir=cache_dir)        else:            ds = load_dataset(repo, config, split=split, cache_dir=cache_dir)    except Exception as exc:  # noqa: BLE001        log(f"  !! {repo}{'/' + config if config else ''}: "            f"{type(exc).__name__}: {str(exc)[:160]} -- skipped")        return []    if cap and len(ds) > cap:        ds = ds.select(range(cap))    out = []    for row in ds:        try:            ex = convert(row)        except Exception:  # noqa: BLE001 - a malformed row is not worth a traceback            ex = None        if ex:            out.append(ex)    name = repo.rsplit("/", 1)[-1][:40] if repo.endswith(".parquet") else repo    log(f"  {name}{'/' + config if config else '':<24} {len(out):>7,} examples")    return outMAX_CHARS_PER_TOKEN = 8   # English runs ~4.7 on this BPE, German ~3.5def _encoded(fmt, convs, seq, chunk=1024):    """Yield `fmt.encode(c)` for the conversations that can possibly fit in `seq`.    ~40% of smol-smoltalk is longer than one block and is dropped after encoding    anyway, and those are the long ones -- about two thirds of its tokens. A    conversation whose text exceeds `seq * MAX_CHARS_PER_TOKEN` characters cannot    come in under `seq` tokens, so it is skipped before the tokenizer sees it.    Everything else goes through `encode_batch` a chunk at a time, in order, so    the corpus is the one the one-at-a-time loop built.    """    limit = seq * MAX_CHARS_PER_TOKEN    for lo in range(0, len(convs), chunk):        part = [c for c in convs[lo:lo + chunk]                if sum(len(m["content"] or "") for m in c) <= limit]        yield from fmt.encode_batch(part)def build_sft_corpus(cfg, tok, log=print):    """Tokenise the whole SFT mixture into (ids uint16, loss uint8) blocks.    Conversations are concatenated into fixed `seq_len + 1` blocks rather than    padded to length: at these sequence lengths padding would waste roughly half    the compute, and this is the same packing the pretraining loader does. The    price is that attention can see across the boundary between two packed    conversations. It is a real effect and a well-known one; with `<|im_start|>`    marking every turn the model learns to reset on it, and buying a document    mask here would cost the fused causal SDPA path that makes the T4 usable.    """    fmt = ChatFormat(tok)    rng = random.Random(cfg["seed"])    by_lang = {"en": [], "de": []}    log("SFT sources:")    for repo, config, split, lang, convert, cap in SFT_SOURCES:        by_lang[lang] += _load_rows(repo, config, split, convert, cap,                                    cfg["hf_cache_dir"], log)    for v in by_lang.values():        rng.shuffle(v)    seq = cfg["seq_len"]    budget = {"de": int(cfg["sft_tokens"] * cfg["de_ratio"]),              "en": int(cfg["sft_tokens"] * (1 - cfg["de_ratio"]))}    ids_out, loss_out = [], []    buf_i: list[int] = []    buf_l: list[int] = []    kept = {"en": 0, "de": 0}    # Interleave the two languages by token count, the same rule the pretraining    # packer uses -- German instruction rows are longer than English ones, so    # alternating by example would not give a 60/40 corpus.    cur = {k: _encoded(fmt, v, seq) for k, v in by_lang.items()}    while True:        lang = "de" if kept["de"] < cfg["de_ratio"] * (kept["en"] + kept["de"]) else "en"        if kept[lang] >= budget[lang]:            other = "en" if lang == "de" else "de"            if kept[other] >= budget[other]:                break            lang = other        ex = next(cur[lang], None)        if ex is None:            log(f"  {lang} exhausted at {kept[lang] / 1e6:.1f}M tokens "                f"(budget {budget[lang] / 1e6:.1f}M)")            budget[lang] = kept[lang]          # source exhausted; let the other fill up            if all(kept[k] >= budget[k] for k in budget):                break            continue        i, l = ex        if len(i) > seq:                       # drop, do not truncate: a conversation            continue                           # cut mid-answer teaches truncated answers        if not any(l):            continue        kept[lang] += len(i)        buf_i += i        buf_l += l        while len(buf_i) >= seq + 1:            ids_out.append(np.asarray(buf_i[:seq + 1], dtype=np.uint16))            loss_out.append(np.asarray(buf_l[:seq + 1], dtype=np.uint8))            del buf_i[:seq + 1], buf_l[:seq + 1]    ids = np.stack(ids_out) if ids_out else np.zeros((0, seq + 1), np.uint16)    loss = np.stack(loss_out) if loss_out else np.zeros((0, seq + 1), np.uint8)    # A block that happens to contain no assistant token at all would make    # cross_entropy average over an empty set and return NaN, which then poisons    # the gradients for the whole accumulation. Rare, but it costs one line.    keep = loss[:, 1:].any(axis=1)    if not keep.all():        log(f"  dropped {int((~keep).sum())} blocks with no assistant token")        ids, loss = ids[keep], loss[keep]    log(f"SFT corpus: {len(ids)} blocks of {seq + 1} = {ids.size / 1e6:.1f}M tokens "        f"({kept['en'] / 1e6:.1f}M EN + {kept['de'] / 1e6:.1f}M DE), "        f"{loss.mean():.1%} of positions carry loss")    return ids, lossdef build_pref_pairs(cfg, tok, log=print):    """-> list of (prompt_ids, chosen_ids, rejected_ids), already length-filtered.    Both responses are stored complete: the reward model scores the last token,    so a truncated response would be scored on a sentence that simply stops.    """    fmt = ChatFormat(tok)    rng = random.Random(cfg["seed"] + 5)    by_lang = {"en": [], "de": []}    log("preference sources:")    for repo, config, split, lang, convert, cap in PREF_SOURCES:        by_lang[lang] += _load_rows(repo, config, split, convert, cap,                                    cfg["hf_cache_dir"], log)    out = []    max_len = cfg["seq_len"]    for lang, rows in by_lang.items():        rng.shuffle(rows)        cap = cfg["pref_pairs"] * (cfg["de_ratio"] if lang == "de" else 1 - cfg["de_ratio"])        n = 0        for msgs, chosen, rejected in rows:            if n >= cap:                break            p = fmt.prompt_ids(msgs)            hdr = len(fmt.head["assistant"])            c = fmt.encode([dict(role="assistant", content=chosen)])[0][hdr:]            r = fmt.encode([dict(role="assistant", content=rejected)])[0][hdr:]            if len(p) + max(len(c), len(r)) > max_len or not c or not r:                continue            out.append((p, c, r))            n += 1        log(f"  {lang}: {n} pairs")    rng.shuffle(out)    return outdef build_prompt_set(cfg, tok, log=print):    """Prompts for GRPO rollouts: the preference prompts, plus SFT user turns.    Reusing the preference prompts is deliberate -- the reward model was fitted    on exactly this prompt distribution, and it is least wrong there.    """    fmt = ChatFormat(tok)    rng = random.Random(cfg["seed"] + 11)    prompts = []    for repo, config, split, lang, convert, cap in PREF_SOURCES:        for item in _load_rows(repo, config, split, convert, cap, cfg["hf_cache_dir"], log):            p = fmt.prompt_ids(item[0])            if cfg["grpo_min_prompt"] <= len(p) <= cfg["grpo_max_prompt"]:                prompts.append(p)    rng.shuffle(prompts)    prompts = prompts[:cfg["grpo_prompts"]]    log(f"GRPO prompt pool: {len(prompts)} prompts, "        f"median length {int(np.median([len(p) for p in prompts])) if prompts else 0}")    return prompts

In [ ]:
%%writefile /kaggle/working/posttrain.py"""Shared runtime for the three post-training stages (SFT, reward model, GRPO).All three are the same machine as `train.py`: two T4s under torchrun, fp16 with aGradScaler because sm75 has no bf16 tensor cores, Muon on the hidden matrices andAdam on everything else, and the same loss-scale floor -- underflowed gradientsare worse with Muon than with Adam, because Newton-Schulz normalises whateverdirection it is handed and turns quantisation noise into a full-size update.Factored out here rather than copied three times, and kept separate from`train.py` so that a change to a post-training stage cannot break pretraining."""from __future__ import annotationsimport datetimeimport jsonimport mathimport osimport sysimport torchimport torch.distributed as distfrom torch.nn.parallel import DistributedDataParallel as DDPsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))import chatdata  # noqa: E402from model import GPT, Muon  # noqa: E402def load_config():    return json.load(open(os.environ.get("POSTTRAIN_CONFIG", "config_post.json")))class Runtime:    """Distributed setup, tokenizer, model loading and the fp16 optimizer step."""    def __init__(self, cfg, stage: str):        self.cfg, self.stage = cfg, stage        self.rank = int(os.environ.get("RANK", 0))        self.local_rank = int(os.environ.get("LOCAL_RANK", 0))        self.world = int(os.environ.get("WORLD_SIZE", 1))        self.master = self.rank == 0        torch.cuda.set_device(self.local_rank)        self.dev = torch.device("cuda", self.local_rank)        if self.world > 1 and not dist.is_initialized():            dist.init_process_group(                "nccl", device_id=self.dev,                timeout=datetime.timedelta(minutes=cfg["nccl_timeout_min"]))        torch.manual_seed(cfg["seed"] + self.rank)        torch.backends.cudnn.benchmark = True        self.metrics_path = os.path.join(cfg["out_dir"], f"{stage}_metrics.jsonl")        from tokenizers import Tokenizer        self.tok = Tokenizer.from_file(cfg["tokenizer_path"])        # Growing the vocabulary has to happen before any model is built, and        # identically on both ranks, or the two will disagree about the shape of        # the embedding and DDP's first all-reduce will fail with a size mismatch.        self.real_vocab = chatdata.ensure_chat_tokens(self.tok, log=self.log)        self.pad_vocab = (self.real_vocab + 127) // 128 * 128        self.eot = self.tok.token_to_id(cfg["eot_token"])        self.fmt = chatdata.ChatFormat(self.tok)        self.log(f"vocab {self.real_vocab} -> padded {self.pad_vocab} | "                 f"im_start {self.fmt.im_start} im_end {self.fmt.im_end}")    # -- logging ----------------------------------------------------------- #    def log(self, msg):        if self.master:            print(msg, flush=True)    def metric(self, **row):        if self.master:            with open(self.metrics_path, "a") as fh:                fh.write(json.dumps(row) + "\n")    def barrier(self):        if self.world > 1:            dist.barrier()    def agree(self, *vals):        """Rank 0's values on every rank, for any decision taken from the wall clock.        Each process has its own clock, started at its own moment, so `time.time() >        deadline` can flip on one rank a step before the other. The rank that leaves        the loop then pairs its evaluation all-reduce with the other's gradient        buckets, NCCL hangs for nccl_timeout_min, and the stage dies without its        final checkpoint -- after which the Kaggle session idles on the quota.        """        if self.world == 1:            return [float(v) for v in vals]        t = torch.tensor([float(v) for v in vals], dtype=torch.float64, device=self.dev)        dist.broadcast(t, src=0)        return t.tolist()    # -- model ------------------------------------------------------------- #    def new_gpt(self, c=None, pad_vocab=None) -> GPT:        cfg = self.cfg        c = c or cfg        return GPT(pad_vocab or self.pad_vocab, c["n_layer"], c["n_head"], c["dim"],                   max(c["seq_len"], cfg["seq_len"]), c["mlp_mult"], c["softcap"],                   c.get("relu2_clamp", 128.0))    def build_model(self, ckpt_path, strict=True) -> GPT:        """Load a pretraining or SFT checkpoint into a fresh GPT.        Handles both blob layouts (`model_final.pt` and the rolling `ckpt.pt`),        and grows the embedding/head if the checkpoint predates the chat tokens.        """        blob = torch.load(ckpt_path, map_location="cpu", weights_only=False)        c = blob.get("cfg", self.cfg)        # Take the vocabulary from the weights, not from `pad_vocab`: the rolling        # `ckpt.pt` a cut-short pretraining session leaves behind has no such key,        # and if the chat tokens have just widened self.pad_vocab, guessing wrong        # means load_state_dict raises a size mismatch that reads like corruption.        ckpt_vocab = blob["model"]["embed.weight"].shape[0]        net = self.new_gpt(c, ckpt_vocab)        missing = net.load_state_dict(blob["model"], strict=False)        if strict and missing.unexpected_keys:            raise RuntimeError(f"unexpected keys in {ckpt_path}: {missing.unexpected_keys[:5]}")        if missing.missing_keys:            self.log(f"  (fresh weights for {len(missing.missing_keys)} tensors: "                     f"{missing.missing_keys[:4]})")        net = chatdata.grow_vocab(net, self.pad_vocab)        net = net.to(self.dev)        self.log(f"loaded {os.path.basename(ckpt_path)}"                 + (f" (step {blob['steps']}, {blob['tokens']/1e9:.2f}B tokens, "                    f"val {blob['val_loss']:.4f})" if "steps" in blob else ""))        return net    @staticmethod    def freeze_half(net):        """fp16 weights, fp32 buffers, no gradients, eval mode.        `Module.half()` would also convert the rotary cos/sin tables, and a        reference model whose angles are fp16 while the policy's are fp32 gives a        systematically biased KL estimate -- so only the parameters are cast.        """        for p in net.parameters():            p.data = p.data.half()            p.requires_grad_(False)        return net.eval()    def frozen_copy(self, net) -> GPT:        """A gradient-free fp16 clone -- the GRPO KL reference. Half precision        saves 250 MB, and it only ever runs under autocast fp16 anyway."""        import copy        return self.freeze_half(copy.deepcopy(net))    def wrap(self, net, compile_ok=True):        model = net        if self.cfg["compile"] and compile_ok:            model = torch.compile(model, mode=self.cfg["compile_mode"])        if self.world > 1:            model = DDP(model, device_ids=[self.local_rank], gradient_as_bucket_view=True,                        broadcast_buffers=False, bucket_cap_mb=40)            if self.cfg["fp16_allreduce"]:                from torch.distributed.algorithms.ddp_comm_hooks import default_hooks                model.register_comm_hook(None, default_hooks.fp16_compress_hook)        return model    # -- optimizers -------------------------------------------------------- #    def build_optimizers(self, hidden, scalar, embed, head, lr_scale=1.0):        cfg = self.cfg        groups = [(embed, cfg["lr_embed"]), (head, cfg["lr_head"]), (scalar, cfg["lr_scalar"])]        adam = torch.optim.Adam(            [dict(params=list(p), lr=lr * lr_scale) for p, lr in groups if list(p)],            betas=tuple(cfg["adam_betas"]), eps=1e-10, fused=True)        muon = Muon(hidden, lr=cfg["lr_muon"] * lr_scale, momentum=cfg["muon_momentum"],                    ns_steps=cfg["ns_steps"], weight_decay=cfg["muon_weight_decay"],                    rank=self.rank, world_size=self.world)        self.optimizers = [adam, muon]        self.base_lrs = [[g["lr"] for g in o.param_groups] for o in self.optimizers]        self.params = [p for o in self.optimizers for g in o.param_groups for p in g["params"]]        self.scaler = torch.amp.GradScaler("cuda", init_scale=2.0**14, growth_interval=500)        self.skipped = 0        return adam, muon    def set_lr(self, mult):        for opt, bases in zip(self.optimizers, self.base_lrs):            for g, base in zip(opt.param_groups, bases):                g["lr"] = base * mult    def lr_mult(self, step, total):        """Warmup then cosine decay to `final_lr_frac`.        Pretraining uses a trapezoid because its length is fitted to a wall clock        it does not know in advance. A post-training run knows its step count up        front, so it can just decay.        """        warm = max(1, self.cfg["warmup_steps"])        if step < warm:            return (step + 1) / warm        prog = min(1.0, (step - warm) / max(1, total - warm))        f = self.cfg["final_lr_frac"]        return f + (1 - f) * 0.5 * (1 + math.cos(math.pi * prog))    def optimizer_step(self):        """Unscale, clip, skip on non-finite gradients, keep the scale in range.        Returns (grad_norm, stepped). Every rank must reach the same verdict, so        the finite check is all-reduced -- one rank stepping while the other skips        desynchronises the weights permanently.        """        for opt in self.optimizers:            self.scaler.unscale_(opt)        gnorm = torch.nn.utils.clip_grad_norm_(self.params, self.cfg["grad_clip"])        ok = torch.isfinite(gnorm).to(torch.float32)        if self.world > 1:            dist.all_reduce(ok, op=dist.ReduceOp.MIN)        stepped = ok.item() == 1.0        if stepped:            for opt in self.optimizers:                opt.step()        else:            self.skipped += 1        self.scaler.update()        cur = self.scaler.get_scale()        lo, hi = self.cfg["min_loss_scale"], self.cfg["max_loss_scale"]        if not lo <= cur <= hi:            self.scaler.update(float(min(max(cur, lo), hi)))        for opt in self.optimizers:            opt.zero_grad(set_to_none=True)        return float(gnorm), stepped    # -- checkpoints ------------------------------------------------------- #    def save(self, net, name, **extra):        """Weights only, written once by the master.        Unlike pretraining this does not persist Muon's momentum. Each stage runs        for a few hours and is restarted rather than resumed mid-epoch, and Muon's        state is sharded, so keeping it would cost ~340 MB per rank per checkpoint        out of Kaggle's 20 GB output quota for something nothing reads.        """        path = os.path.join(self.cfg["out_dir"], name)        if not self.master:            return path        torch.save(dict(model=net.state_dict(), cfg=self.cfg, pad_vocab=self.pad_vocab,                        real_vocab=self.real_vocab, **extra), path + ".tmp")        os.replace(path + ".tmp", path)        return path    def finish(self):        if self.world > 1:            dist.barrier()            dist.destroy_process_group()

In [ ]:
%%writefile /kaggle/working/sft.py"""Stage 1 of post-training: supervised fine-tuning on bilingual EN/DE chat.Launch:  torchrun --standalone --nproc_per_node=2 sft.pyLoss is computed on assistant content only. The prompt, the role headers and theuser's text are masked out with -100, which `F.cross_entropy` drops from both thenumerator and the denominator -- training on the headers would teach the model toemit "<|im_start|>user", which is the one continuation it must never produce.The corpus is tokenised once into fixed-length packed blocks (chatdata.py) andcached, so this loop has no tokeniser and no network in it: every step is GPUwork, and a second epoch costs nothing extra."""from __future__ import annotationsimport contextlibimport jsonimport mathimport osimport timeimport numpy as npimport torchimport chatdataimport posttrainfrom model import generate_batchCFG = posttrain.load_config()rt = posttrain.Runtime(CFG, "sft")# --------------------------------------------------------------------------- ## corpus# --------------------------------------------------------------------------- ## The fingerprint is in the filename, not just the sizes: changing de_ratio alone# leaves every other part of the key identical, and a cached corpus from before the# change would be reused without a word.CORPUS = os.path.join(CFG["cache_dir"],                      f"sft_{CFG['sft_tokens']}_{rt.real_vocab}_{CFG['seq_len']}_"                      f"{chatdata.sft_sources_key(CFG)}.npz")if not os.path.exists(CORPUS):    # Built by rank 0 only: two ranks downloading the same 700 MB and tokenising    # it twice is minutes of the budget, and np.savez is not atomic.    if rt.master:        ids_np, loss_np = chatdata.build_sft_corpus(CFG, rt.tok, log=rt.log)        np.savez(CORPUS + ".tmp.npz", ids=ids_np, loss=loss_np)        os.replace(CORPUS + ".tmp.npz", CORPUS)        del ids_np, loss_np    else:        # Wait for the file, not in the barrier: a barrier is an NCCL collective,        # and the watchdog kills a rank whose collective is pending for        # nccl_timeout_min -- which is exactly how long this build can take (it        # tokenises ~450M tokens on rank 0). If rank 0 dies, torchrun ends this one.        while not os.path.exists(CORPUS):            time.sleep(5)    rt.barrier()_blob = np.load(CORPUS)ids_all, loss_all = _blob["ids"], _blob["loss"]n_val = min(CFG["sft_val_blocks"], len(ids_all) // 20)val_ids = torch.from_numpy(ids_all[:n_val].astype(np.int64))val_loss = torch.from_numpy(loss_all[:n_val].astype(np.bool_))train_ids = torch.from_numpy(ids_all[n_val:].astype(np.int64))train_loss = torch.from_numpy(loss_all[n_val:].astype(np.bool_))del _blob, ids_all, loss_allMICRO = CFG["micro_bs"]ACCUM = CFG["grad_accum"]PER_STEP = MICRO * ACCUM * rt.worldsteps_per_epoch = len(train_ids) // PER_STEPtotal_steps = max(1, steps_per_epoch * CFG["sft_epochs"])rt.log(f"{len(train_ids)} train blocks + {n_val} val | {steps_per_epoch} steps/epoch x "       f"{CFG['sft_epochs']} epochs = {total_steps} steps "       f"({total_steps * PER_STEP * CFG['seq_len'] / 1e6:.0f}M tokens seen)")# --------------------------------------------------------------------------- ## model# --------------------------------------------------------------------------- #net = rt.build_model(os.path.join(CFG["out_dir"], CFG["base_checkpoint"]))hidden, scalar, embed, head = net.param_groups()rt.build_optimizers(hidden, scalar, embed, head)model = rt.wrap(net)rt.log(f"params {sum(p.numel() for p in net.parameters())/1e6:.1f}M | "       f"lr_muon {CFG['lr_muon']} lr_head {CFG['lr_head']}")def batch_at(source_ids, source_loss, index):    x = source_ids[index].to(rt.dev, non_blocking=True)    m = source_loss[index].to(rt.dev, non_blocking=True)    # -100 is cross_entropy's ignore_index: those positions leave the mean    # entirely, so the reported loss is per *assistant* token, not per token.    return x[:, :-1], torch.where(m[:, 1:], x[:, 1:], -100)@torch.no_grad()def evaluate():    """Every rank must run the same number of batches -- the means are combined    with an AVG all-reduce, so an uneven split silently reweights the result."""    model.eval()    losses = []    for j in range(n_val // (rt.world * MICRO)):        lo = (j * rt.world + rt.rank) * MICRO        inp, tgt = batch_at(val_ids, val_loss, slice(lo, lo + MICRO))        with torch.autocast("cuda", dtype=torch.float16):            losses.append(model(inp, tgt).float())    out = torch.stack(losses).mean() if losses else torch.zeros((), device=rt.dev)    if rt.world > 1:        torch.distributed.all_reduce(out, op=torch.distributed.ReduceOp.AVG)    model.train()    return out.item()# --------------------------------------------------------------------------- ## train# --------------------------------------------------------------------------- #gen = torch.Generator().manual_seed(CFG["seed"])deadline = time.time() + CFG["time_budget_h"] * 3600best_val = float("inf")model.train()step = 0order = torch.randperm(len(train_ids), generator=gen)cursor = 0t0 = time.time()while step < total_steps:    t_step = time.time()    mult = rt.lr_mult(step, total_steps)    rt.set_lr(mult)    for micro in range(ACCUM):        if cursor + MICRO * rt.world > len(order):            # Reseeded identically on every rank: `gen` has seen exactly the same            # call sequence everywhere, so the permutations stay in lockstep.            order, cursor = torch.randperm(len(train_ids), generator=gen), 0        # Ranks take disjoint slices of the same permutation, so an epoch is a        # real epoch across both GPUs rather than each rank seeing everything.        take = order[cursor:cursor + MICRO * rt.world][rt.rank::rt.world]        cursor += MICRO * rt.world        inp, tgt = batch_at(train_ids, train_loss, take)        sync_ctx = (model.no_sync() if (rt.world > 1 and micro < ACCUM - 1)                    else contextlib.nullcontext())        with sync_ctx:            with torch.autocast("cuda", dtype=torch.float16):                loss = model(inp, tgt)            rt.scaler.scale(loss / ACCUM).backward()    gnorm, stepped = rt.optimizer_step()    dt = time.time() - t_step    if step % CFG["log_every"] == 0 or step == total_steps - 1:        tps = PER_STEP * CFG["seq_len"] / dt        rt.log(f"step {step:5d}/{total_steps} loss {loss.item():6.4f} lr {mult:5.3f} "               f"gnorm {gnorm:6.3f} scale {rt.scaler.get_scale():>7.0f} "               f"{dt*1000:6.0f}ms {tps:7.0f} tok/s "               f"eta {(total_steps-step)*dt/3600:4.2f}h")        rt.metric(step=step, loss=loss.item(), lr_mult=mult, tok_s=tps)    if CFG["val_every"] and step > 0 and step % CFG["val_every"] == 0:        vl = evaluate()        mark = ""        if math.isfinite(vl) and vl < best_val:            best_val, mark = vl, "  <- best"            rt.save(net, "sft_best.pt", steps=step, val_loss=vl)        rt.log(f"    val {vl:.4f}  ppl {math.exp(min(vl, 20)):.2f}{mark}")        rt.metric(step=step, val_loss=vl, best_val=best_val)    step += 1    # rank 0 decides -- each process has its own clock (Runtime.agree)    if rt.agree(time.time() > deadline)[0]:        rt.log(f"wall-clock budget reached at step {step}")        breakvl = evaluate()BEST = os.path.join(CFG["out_dir"], "sft_best.pt")if math.isfinite(best_val) and vl > best_val + 0.02 and os.path.exists(BEST):    # `sft_final.pt` is what stages 2 and 3 read, so it must not be whatever the    # last step happened to leave behind. train.py has had this guard since a    # divergence ate a good model; SFT needs it for a duller reason -- two epochs    # over a 150M-token corpus is exactly where a 125M model starts memorising,    # and the second epoch can end worse than it began.    rt.log(f"final state ({vl:.4f}) is worse than the best checkpoint ({best_val:.4f}); "           f"exporting the best one instead")    rt.barrier()          # written by the master alone; both ranks read it back    net.load_state_dict(torch.load(BEST, map_location=rt.dev, weights_only=False)["model"])    vl = evaluate()rt.log(f"done: {step} steps in {(time.time()-t0)/3600:.2f}h | "       f"{rt.skipped} fp16-overflow steps skipped | val {vl:.4f}")# The fingerprint of the mixture this was trained on, next to the checkpoint and# not inside it: the environment cell has to read it on every Run All, and loading# a 516 MB blob to compare twelve characters is not a thing to do at startup.if rt.master:    with open(os.path.join(CFG["out_dir"], "sft_sources.json"), "w") as fh:        json.dump(dict(sources=chatdata.sft_sources_key(CFG),                       sft_tokens=CFG["sft_tokens"], de_ratio=CFG["de_ratio"]), fh)rt.save(net, "sft_final.pt", steps=step, val_loss=vl, tokens=step * PER_STEP * CFG["seq_len"])# --------------------------------------------------------------------------- ## samples -- the point of the stage is that the model now answers, so show it# --------------------------------------------------------------------------- #if rt.master:    msgs = [[dict(role="user", content=p)] for p in CFG["sample_prompts"]]    heads = [rt.fmt.prompt_ids(m) for m in msgs]    ids, _ = generate_batch(net, heads, CFG["sample_tokens"], rt.eot, pad_id=rt.eot,                            stop_ids=(rt.fmt.im_end,), real_vocab=rt.real_vocab,                            temperature=0.8, top_p=0.92, device=rt.dev)    plen = max(len(h) for h in heads)     # generate_batch left-pads to this    for msg, row in zip(msgs, ids):        rt.log(f"\n--- {msg[0]['content']!r}\n"               f"{rt.tok.decode(row.tolist()[plen:]).strip()}\n")rt.finish()

In [ ]:
import json, os, subprocess, sys, timeimport torchSESSION_START = time.time()SESSION_LIMIT_H = 11.5          # Kaggle stops a session at 12hRESERVED_FOR_LATER_CELLS_H = 0.3def session_left_h():    return SESSION_LIMIT_H - (time.time() - SESSION_START) / 3600def trainable_hours_left():    return session_left_h() - RESERVED_FOR_LATER_CELLS_Hfor directory in (CONFIG["out_dir"], CONFIG["cache_dir"], POST["hf_cache_dir"]):    os.makedirs(directory, exist_ok=True)sys.path.insert(0, CONFIG["out_dir"])def have(name):    return os.path.exists(os.path.join(CONFIG["out_dir"], name))def pretraining_finished():    if not (have("model_final.pt") and have("ckpt.pt")):        return False    blob = torch.load(CONFIG["out_dir"] + "/ckpt.pt", map_location="cpu", weights_only=False)    step, total, elapsed = blob["step"] + 1, blob["total_steps"], blob.get("elapsed_h", 0.0)    del blob    print(f"checkpoint: step {step} of {total}, {elapsed:.2f}h of {CONFIG['plan_h']}h trained")    return step >= total and elapsed >= CONFIG["plan_h"] - 0.5if torch.cuda.device_count() != 2:    raise RuntimeError(f"needs the T4 x2 accelerator, found {torch.cuda.device_count()} GPU(s)")DO_PRETRAIN = not pretraining_finished()DO_SFT = not have("sft_final.pt")print(f"{torch.cuda.get_device_name(0)} x2 | pretrain: {DO_PRETRAIN} | fine-tune: {DO_SFT}")print(f"{session_left_h():.1f}h of session clock")

In [ ]:
import importlib, randomimport dataio; importlib.reload(dataio)if DO_PRETRAIN:    rng = random.Random(0)    train_by_source, val_by_source = {}, {}    for name in CONFIG["en_sources"] + CONFIG["de_sources"]:        files = dataio.source_files(name)        rng.shuffle(files)        n_val = min(CONFIG["val_files_per_source"], max(0, len(files) - 1))        val_by_source[name] = files[:n_val]        train_by_source[name] = files[n_val:]        print(f"  {name:15s} {dataio.SOURCES[name]['lang']}  {len(files):4d} parquet, "              f"{n_val} held out for validation")    CONFIG["en_files_by_source"] = {n: train_by_source[n] for n in CONFIG["en_sources"]}    CONFIG["de_files_by_source"] = {n: train_by_source[n] for n in CONFIG["de_sources"]}    CONFIG["val_files_by_source"] = val_by_source    for lang, key in (("en", "anneal_en_files_by_source"),                      ("de", "anneal_de_files_by_source")):        picked = {n: train_by_source[n] for n in CONFIG["anneal_sources"]                  if dataio.SOURCES[n]["lang"] == lang}        CONFIG[key] = picked or CONFIG[f"{lang}_files_by_source"]    for label, names, weights in (("source_weights",                                   CONFIG["en_sources"] + CONFIG["de_sources"],                                   CONFIG["source_weights"]),                                  ("anneal_source_weights", CONFIG["anneal_sources"],                                   CONFIG["anneal_source_weights"])):        missing = [n for n in names if n not in weights]        if missing:            raise RuntimeError(f"{label} has no entry for {missing}")    with open(CONFIG["out_dir"] + "/config.json", "w") as fh:        json.dump(CONFIG, fh, indent=2)    print(f"\n{sum(len(v) for v in train_by_source.values())} training files, "          f"de_ratio {CONFIG['de_ratio']} by tokens")with open(CONFIG["out_dir"] + "/config_post.json", "w") as fh:    json.dump(POST, fh, indent=2)

In [ ]:
import importlib, timeimport tokenizer_train; importlib.reload(tokenizer_train)if DO_PRETRAIN:    t0 = time.time()    tok = tokenizer_train.train(CONFIG, CONFIG["tokenizer_path"])    print(f"\nvocab {tok.get_vocab_size()} ({time.time() - t0:.0f}s)")    CONFIG["chars_per_token"] = tokenizer_train.report(tok, CONFIG).get(        "mixed", CONFIG["chars_per_token"])elif not have("tokenizer.json"):    raise RuntimeError("tokenizer.json is missing and stage 0 is not running")

In [ ]:
def run_stage(script, config_env, config_path, **extra_env):    env = {**os.environ,           config_env: config_path,           "OMP_NUM_THREADS": "1",           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",           "TORCHINDUCTOR_CACHE_DIR": CONFIG["out_dir"] + "/inductor",           **extra_env}    t0 = time.time()    proc = subprocess.Popen(        [sys.executable, "-m", "torch.distributed.run", "--standalone",         "--nproc_per_node=2", script],        cwd=CONFIG["out_dir"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,        text=True, bufsize=1)    for line in proc.stdout:        print(line, end="")    proc.wait()    print(f"\n{script} exited with {proc.returncode} after {(time.time() - t0) / 60:.1f} min")    if proc.returncode != 0:        raise RuntimeError(f"{script} failed (exit {proc.returncode})")if DO_PRETRAIN:    CONFIG["session_budget_h"] = round(max(0.2, trainable_hours_left()), 3)    print(f"training for up to {CONFIG['session_budget_h']:.2f}h, "          f"{CONFIG['plan_h']}h of plan in total")    with open(CONFIG["out_dir"] + "/config.json", "w") as fh:        json.dump(CONFIG, fh, indent=2)    run_stage("train.py", "TRAIN_CONFIG", CONFIG["out_dir"] + "/config.json",              TOKENIZERS_PARALLELISM="false")    plan_finished_in_this_session = pretraining_finished()    DO_PRETRAIN = not plan_finished_in_this_sessionelse:    print("pretraining already finished")

In [ ]:
if DO_SFT and have("model_final.pt") and not DO_PRETRAIN:    budget = round(max(0.2, min(POST["time_budget_h"], trainable_hours_left())), 3)    if budget < 1.0:        print(f"only {budget:.2f}h left of the session - start a new one for stage 1")    else:        POST["time_budget_h"] = budget        print(f"fine-tuning for up to {budget:.2f}h")        with open(CONFIG["out_dir"] + "/config_post.json", "w") as fh:            json.dump(POST, fh, indent=2)        run_stage("sft.py", "POSTTRAIN_CONFIG", CONFIG["out_dir"] + "/config_post.json",                  HF_HOME=POST["hf_cache_dir"], TOKENIZERS_PARALLELISM="true")elif DO_PRETRAIN:    print("stage 1 runs once pretraining has finished its plan")else:    print("sft_final.pt is already there")

In [ ]:
print(f"""Done for this session. {"Pretraining has hours left - press Run All again in a new session."                        if DO_PRETRAIN else "sft_final.pt is the chat model."}Stop the session by hand (Run -> Stop Session): Kaggle charges the weekly quota foran idle session until it is stopped or reaches its 12h cap.""")